In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive
import os
os.makedirs('/content/drive/MyDrive/UttarakhandDisasterSystem', exist_ok=True)

# Change working directory to Drive folder
os.chdir('/content/drive/MyDrive/UttarakhandDisasterSystem')
print(f"Working in: {os.getcwd()}")

Mounted at /content/drive
Working in: /content/drive/MyDrive/UttarakhandDisasterSystem


In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import os
import datetime
from io import StringIO

print("Libraries ready ✅")

LAT_MIN    = 28.0
LAT_MAX    = 32.0
LON_MIN    = 77.0
LON_MAX    = 81.0
DATE_START = "20250301"
DATE_END   = "20260301"
BASE_URL   = "https://power.larc.nasa.gov/api/temporal/daily/regional"
OUTPUT_DIR = "nasa_power_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PARAMETERS = {
    "RH2M"              : "Relative Humidity at 2 Meters (%)",
    "T2M"               : "Temperature at 2 Meters (°C)",
    "PRECTOTCORR"       : "Precipitation Corrected (mm/day)",
    "WS10M"             : "Wind Speed at 10 Meters (m/s)",
    "T2M_MAX"           : "Maximum Temperature at 2 Meters (°C)",
    "T2M_MIN"           : "Minimum Temperature at 2 Meters (°C)",
    "GWETROOT"          : "Root Zone Soil Wetness (0–1 fraction)",
    "ALLSKY_SFC_SW_DWN" : "Solar Radiation (kWh/m²/day)",
}

print(f"Config ready ✅")
print(f"Region  : LAT {LAT_MIN}–{LAT_MAX}N  |  LON {LON_MIN}–{LON_MAX}E")
print(f"Period  : {DATE_START} → {DATE_END}")
print(f"Output  : ./{OUTPUT_DIR}/")


def parse_nasa_csv(raw_text, param_code):
    """
    NASA POWER API returns DOY (day of year) instead of MO + DY.
    This parser handles both formats and always outputs a clean
    DataFrame with: LAT, LON, YEAR, MO, DY, DATE, <param_code>
    """
    lines = raw_text.strip().split('\n')

    # ── Find where data starts (after -END HEADER-) ──
    data_start = 0
    for i, line in enumerate(lines):
        if line.strip() == "-END HEADER-":
            data_start = i + 1
            break

    if data_start == 0:
        # No header block — might be pure CSV already
        data_start = 0

    csv_text = "\n".join(lines[data_start:])
    df = pd.read_csv(StringIO(csv_text))

    # ── Replace NASA missing value sentinel ──
    df[param_code] = pd.to_numeric(df[param_code], errors='coerce')
    df[param_code] = df[param_code].replace(-999, np.nan)

    # ── Handle DOY format (API) vs MO+DY format (manual download) ──
    if 'DOY' in df.columns and 'MO' not in df.columns:
        # API format: convert DOY → DATE → MO, DY
        def doy_to_date(row):
            base = datetime.date(int(row['YEAR']), 1, 1)
            return base + datetime.timedelta(days=int(row['DOY']) - 1)

        df['DATE'] = pd.to_datetime(df.apply(doy_to_date, axis=1))
        df['MO']   = df['DATE'].dt.month
        df['DY']   = df['DATE'].dt.day

    elif 'MO' in df.columns and 'DY' in df.columns:
        # Manual download format: build DATE from YEAR, MO, DY
        df['DATE'] = pd.to_datetime(df[['YEAR', 'MO', 'DY']].rename(
            columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day'}
        ))
    else:
        raise ValueError(f"Unexpected columns: {list(df.columns)}")

    # ── Keep only essential columns ──
    keep = ['LAT', 'LON', 'YEAR', 'MO', 'DY', 'DATE', param_code]
    df = df[[c for c in keep if c in df.columns]].copy()

    return df

#DOWNLOADS
def download_parameter(param_code, description, retries=3, wait=15):
    output_path = os.path.join(OUTPUT_DIR, f"nasa_{param_code}.csv")

    # ── Skip if already downloaded ──
    if os.path.exists(output_path):
        print(f"  ⏭  {param_code:<22} already exists — loading from disk")
        with open(output_path, 'r') as f:
            raw = f.read()
        return parse_nasa_csv(raw, param_code)

    api_params = {
        "parameters"    : param_code,
        "community"     : "AG",
        "format"        : "CSV",
        "start"         : DATE_START,
        "end"           : DATE_END,
        "latitude-min"  : LAT_MIN,
        "latitude-max"  : LAT_MAX,
        "longitude-min" : LON_MIN,
        "longitude-max" : LON_MAX,
        "time-standard" : "LST",
    }

    for attempt in range(1, retries + 1):
        try:
            print(f"  ↓  {param_code:<22} Downloading... (attempt {attempt}/{retries})")
            resp = requests.get(BASE_URL, params=api_params, timeout=180)
            resp.raise_for_status()

            raw_text = resp.text

            # ── Check for error messages in response body ──
            if "Error" in raw_text[:500] or "error" in raw_text[:500]:
                print(f"  ⚠  API returned an error message:")
                print(f"     {raw_text[:300]}")
                raise ValueError("API error in response body")

            # ── Save raw file ──
            with open(output_path, 'w') as f:
                f.write(raw_text)

            # ── Parse ──
            df = parse_nasa_csv(raw_text, param_code)

            size_kb = os.path.getsize(output_path) / 1024
            print(f"  ✅ {param_code:<22} {len(df):>6} rows  ({size_kb:.1f} KB)")
            return df

        except Exception as e:
            print(f"  ⚠  Error: {e}")
            if attempt < retries:
                print(f"     Waiting {wait}s before retry...")
                time.sleep(wait)

    print(f"  ✖  FAILED: {param_code} after {retries} attempts")
    return None


print("=" * 60)
print("  NASA POWER — DOWNLOADING ALL PARAMETERS")
print("=" * 60)

downloaded = {}
failed     = []

for i, (param_code, description) in enumerate(PARAMETERS.items(), 1):
    print(f"\n[{i}/{len(PARAMETERS)}] {description}")
    df = download_parameter(param_code, description)
    if df is not None:
        downloaded[param_code] = df
    else:
        failed.append(param_code)
    time.sleep(3)   # be polite to NASA servers

print("\n" + "=" * 60)
print(f"  Downloaded : {len(downloaded)}/{len(PARAMETERS)} parameters")
if failed:
    print(f"  Failed     : {', '.join(failed)}")
print("=" * 60)

#PREPARING THE MASTER DATASET
def merge_all(downloaded_dict):
    master = None
    for param_code, df in downloaded_dict.items():
        if master is None:
            master = df.copy()
        else:
            merge_cols = ['LAT', 'LON', 'YEAR', 'MO', 'DY', 'DATE']
            merge_cols = [c for c in merge_cols if c in master.columns and c in df.columns]
            master = master.merge(df, on=merge_cols, how='outer')

    master = master.sort_values(['DATE', 'LAT', 'LON']).reset_index(drop=True)
    return master

if downloaded:
    df_master = merge_all(downloaded)

    print(f"Master DataFrame shape : {df_master.shape}")
    print(f"Columns                : {list(df_master.columns)}")
    print(f"Date range             : {df_master['DATE'].min().date()} → {df_master['DATE'].max().date()}")
    print(f"Grid                   : {df_master['LAT'].nunique()} lats × {df_master['LON'].nunique()} lons")

    print(f"\nMissing value %:")
    for p in PARAMETERS:
        if p in df_master.columns:
            pct = 100 * df_master[p].isna().sum() / len(df_master)
            flag = "✅" if pct < 1 else "⚠ "
            print(f"  {flag} {p:<22} : {pct:.1f}% missing")

    print(f"\nFirst 5 rows:")
    print(df_master.head())
else:
    print("Nothing to merge.")

if downloaded:
    master_csv  = os.path.join(OUTPUT_DIR, "uttarakhand_master.csv")
    master_json = os.path.join(OUTPUT_DIR, "uttarakhand_master.json")

    df_master.to_csv(master_csv, index=False)
    df_master.to_json(master_json, orient="records", date_format="iso", indent=2)

    print(f"\nFiles saved in ./{OUTPUT_DIR}/:")
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
        print(f"  {fname:<45} ({size:.1f} KB)")

    print(f"\n✅  All done! Run Step 2 preprocessing next.")

Libraries ready ✅
Config ready ✅
Region  : LAT 28.0–32.0N  |  LON 77.0–81.0E
Period  : 20250301 → 20260301
Output  : ./nasa_power_data/
  NASA POWER — DOWNLOADING ALL PARAMETERS

[1/8] Relative Humidity at 2 Meters (%)
  ⏭  RH2M                   already exists — loading from disk

[2/8] Temperature at 2 Meters (°C)
  ⏭  T2M                    already exists — loading from disk

[3/8] Precipitation Corrected (mm/day)
  ⏭  PRECTOTCORR            already exists — loading from disk

[4/8] Wind Speed at 10 Meters (m/s)
  ⏭  WS10M                  already exists — loading from disk

[5/8] Maximum Temperature at 2 Meters (°C)
  ⏭  T2M_MAX                already exists — loading from disk

[6/8] Minimum Temperature at 2 Meters (°C)
  ⏭  T2M_MIN                already exists — loading from disk

[7/8] Root Zone Soil Wetness (0–1 fraction)
  ⏭  GWETROOT               already exists — loading from disk

[8/8] Solar Radiation (kWh/m²/day)
  ⏭  ALLSKY_SFC_SW_DWN      already exists — loading from 

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
from scipy.interpolate import griddata

print("All imports OK")

All imports OK


In [ ]:
import pandas as pd
import numpy as np
import os
import json

print("Imports ready ✅")
df = pd.read_csv("nasa_power_data/uttarakhand_master.csv", parse_dates=["DATE"])

print(f"Loaded : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

print("Missing values BEFORE cleaning:")
for col in ["RH2M","T2M","PRECTOTCORR","WS10M","T2M_MAX","T2M_MIN","GWETROOT","ALLSKY_SFC_SW_DWN"]:
    pct = 100 * df[col].isna().sum() / len(df)
    print(f"  {col:<22}: {pct:.1f}%")

# ── Drop ALLSKY (75% missing — not usable) ──
df.drop(columns=["ALLSKY_SFC_SW_DWN"], inplace=True)
print("\nDropped ALLSKY_SFC_SW_DWN (75% missing)")

# ── Sort so interpolation works correctly per grid cell ──
df = df.sort_values(["LAT", "LON", "DATE"]).reset_index(drop=True)

# ── Interpolate missing values within each grid cell ──
PARAMS = ["RH2M", "T2M", "PRECTOTCORR", "WS10M", "T2M_MAX", "T2M_MIN", "GWETROOT"]

for param in PARAMS:
    df[param] = (
        df.groupby(["LAT", "LON"])[param]
          .transform(lambda x: x.interpolate(method="linear", limit_direction="both"))
    )

# ── If still any NaN (isolated grid cells) → fill with column median ──
for param in PARAMS:
    median_val = df[param].median()
    df[param] = df[param].fillna(median_val)

print("\nMissing values AFTER cleaning:")
for param in PARAMS:
    pct = 100 * df[param].isna().sum() / len(df)
    status = "✅" if pct == 0 else "⚠ "
    print(f"  {status} {param:<22}: {pct:.1f}%")

# ── Day of year & week ──
df["DAY_OF_YEAR"] = df["DATE"].dt.dayofyear
df["WEEK"]        = df["DATE"].dt.isocalendar().week.astype(int)
df["MONTH_NAME"]  = df["DATE"].dt.strftime("%b")

# ── Indian meteorological seasons ──
def get_season(month):
    if month in [12, 1, 2]:       return "Winter"
    elif month in [3, 4, 5]:      return "Pre-Monsoon"
    elif month in [6, 7, 8, 9]:   return "Monsoon"
    else:                          return "Post-Monsoon"

df["SEASON"] = df["MO"].apply(get_season)

# ── Grid cell label ──
df["GRID_CELL"] = df["LAT"].astype(str) + "N_" + df["LON"].astype(str) + "E"

print("Season distribution:")
print(df["SEASON"].value_counts().to_string())


df = df.sort_values(["LAT", "LON", "DATE"]).reset_index(drop=True)

def rolling(series, window, min_p=1):
    return series.rolling(window, min_periods=min_p).mean().round(3)

grp = df.groupby(["LAT", "LON"])

# ── 7-day rolling means ──
df["RH2M_7D"]        = grp["RH2M"].transform(lambda x: rolling(x, 7))
df["PRECIP_7D"]      = grp["PRECTOTCORR"].transform(lambda x: rolling(x, 7))
df["WIND_7D"]        = grp["WS10M"].transform(lambda x: rolling(x, 7))
df["SOIL_7D"]        = grp["GWETROOT"].transform(lambda x: rolling(x, 7))

# ── 30-day rolling means ──
df["RH2M_30D"]       = grp["RH2M"].transform(lambda x: rolling(x, 30, min_p=7))
df["PRECIP_30D"]     = grp["PRECTOTCORR"].transform(lambda x: rolling(x, 30, min_p=7))
df["TEMP_30D"]       = grp["T2M"].transform(lambda x: rolling(x, 30, min_p=7))

# ── Anomalies (today vs 30-day baseline) ──
df["TEMP_ANOMALY"]   = (df["T2M"]          - df["TEMP_30D"]).round(3)
df["RH2M_ANOMALY"]   = (df["RH2M"]         - df["RH2M_30D"]).round(3)
df["PRECIP_ANOMALY"] = (df["PRECTOTCORR"]  - df["PRECIP_30D"]).round(3)

# ── Cumulative 3-day precipitation (flood trigger) ──
df["PRECIP_3D_CUM"]  = grp["PRECTOTCORR"].transform(lambda x: x.rolling(3, min_periods=1).sum()).round(3)

print("Rolling features added ✅")
print(f"New columns: {[c for c in df.columns if any(x in c for x in ['7D','30D','ANOMALY','CUM'])]}")

def fire_risk(row):
    score = 0

    # Humidity contribution (most important)
    if row["RH2M"] < 20:       score += 4
    elif row["RH2M"] < 30:     score += 3
    elif row["RH2M"] < 45:     score += 2
    elif row["RH2M"] < 60:     score += 1

    # Temperature contribution
    if row["T2M"] > 35:        score += 3
    elif row["T2M"] > 28:      score += 2
    elif row["T2M"] > 22:      score += 1

    # Wind contribution
    if row["WS10M"] > 8:       score += 2
    elif row["WS10M"] > 5:     score += 1

    # Soil dryness
    if row["GWETROOT"] < 0.1:  score += 2
    elif row["GWETROOT"] < 0.2:score += 1

    # Season multiplier
    if row["SEASON"] == "Pre-Monsoon":  score = int(score * 1.3)
    elif row["SEASON"] == "Winter":     score = int(score * 1.1)
    elif row["SEASON"] == "Monsoon":    score = int(score * 0.3)

    # Cap and label
    score = min(score, 10)
    if score >= 8:    return "EXTREME"
    elif score >= 6:  return "HIGH"
    elif score >= 4:  return "MODERATE"
    elif score >= 2:  return "LOW"
    else:             return "MINIMAL"

df["FIRE_RISK"] = df.apply(fire_risk, axis=1)

def flood_risk(row):
    score = 0

    # Daily rainfall
    if row["PRECTOTCORR"] > 100:   score += 4
    elif row["PRECTOTCORR"] > 50:  score += 3
    elif row["PRECTOTCORR"] > 20:  score += 2
    elif row["PRECTOTCORR"] > 10:  score += 1

    # 3-day cumulative rain
    if row["PRECIP_3D_CUM"] > 150: score += 3
    elif row["PRECIP_3D_CUM"] > 80:score += 2
    elif row["PRECIP_3D_CUM"] > 40:score += 1

    # 7-day mean rainfall
    if row["PRECIP_7D"] > 30:      score += 2
    elif row["PRECIP_7D"] > 15:    score += 1

    # Soil already saturated
    if row["GWETROOT"] > 0.8:      score += 2
    elif row["GWETROOT"] > 0.6:    score += 1

    # Season factor
    if row["SEASON"] == "Monsoon":      score = int(score * 1.4)
    elif row["SEASON"] == "Post-Monsoon": score = int(score * 1.1)

    score = min(score, 10)
    if score >= 8:    return "EXTREME"
    elif score >= 6:  return "HIGH"
    elif score >= 4:  return "MODERATE"
    elif score >= 2:  return "LOW"
    else:             return "MINIMAL"

df["FLOOD_RISK"] = df.apply(flood_risk, axis=1)

def landslide_risk(row):
    score = 0

    # 7-day mean rainfall (sustained rain = key trigger)
    if row["PRECIP_7D"] > 40:      score += 4
    elif row["PRECIP_7D"] > 25:    score += 3
    elif row["PRECIP_7D"] > 15:    score += 2
    elif row["PRECIP_7D"] > 8:     score += 1

    # Soil wetness (saturated soil = unstable slopes)
    if row["GWETROOT"] > 0.85:     score += 3
    elif row["GWETROOT"] > 0.7:    score += 2
    elif row["GWETROOT"] > 0.55:   score += 1

    # 7-day mean humidity (persistently wet air)
    if row["RH2M_7D"] > 85:        score += 2
    elif row["RH2M_7D"] > 70:      score += 1

    # Today's rainfall (triggering event on saturated soil)
    if row["PRECTOTCORR"] > 50:    score += 2
    elif row["PRECTOTCORR"] > 20:  score += 1

    # Higher latitude = higher elevation = more risk
    if row["LAT"] >= 30.5:         score += 1

    # Season factor
    if row["SEASON"] == "Monsoon":        score = int(score * 1.5)
    elif row["SEASON"] == "Post-Monsoon": score = int(score * 1.2)

    score = min(score, 10)
    if score >= 8:    return "EXTREME"
    elif score >= 6:  return "HIGH"
    elif score >= 4:  return "MODERATE"
    elif score >= 2:  return "LOW"
    else:             return "MINIMAL"

df["LANDSLIDE_RISK"] = df.apply(landslide_risk, axis=1)

def heatwave_risk(row):
    score = 0

    # Max temperature (adjusted for Uttarakhand hills)
    if row["T2M_MAX"] > 38:        score += 4
    elif row["T2M_MAX"] > 33:      score += 3
    elif row["T2M_MAX"] > 28:      score += 2
    elif row["T2M_MAX"] > 24:      score += 1

    # Warm nights (T2M_MIN)
    if row["T2M_MIN"] > 25:        score += 2
    elif row["T2M_MIN"] > 20:      score += 1

    # Temperature anomaly (hotter than usual)
    if row["TEMP_ANOMALY"] > 5:    score += 2
    elif row["TEMP_ANOMALY"] > 3:  score += 1

    # Low humidity amplifies heat stress
    if row["RH2M"] < 20:           score += 1

    # Season factor
    if row["SEASON"] == "Pre-Monsoon":  score = int(score * 1.4)
    elif row["SEASON"] == "Monsoon":    score = int(score * 0.5)
    elif row["SEASON"] == "Winter":     score = int(score * 0.2)

    score = min(score, 10)
    if score >= 8:    return "EXTREME"
    elif score >= 6:  return "HIGH"
    elif score >= 4:  return "MODERATE"
    elif score >= 2:  return "LOW"
    else:             return "MINIMAL"

df["HEATWAVE_RISK"] = df.apply(heatwave_risk, axis=1)

# ── Numeric versions (for ML in Step 4) ──
risk_map = {"MINIMAL": 0, "LOW": 1, "MODERATE": 2, "HIGH": 3, "EXTREME": 4}
df["FIRE_RISK_NUM"]      = df["FIRE_RISK"].map(risk_map)
df["FLOOD_RISK_NUM"]     = df["FLOOD_RISK"].map(risk_map)
df["LANDSLIDE_RISK_NUM"] = df["LANDSLIDE_RISK"].map(risk_map)
df["HEATWAVE_RISK_NUM"]  = df["HEATWAVE_RISK"].map(risk_map)

# ── Overall max risk (worst hazard of the day for that cell) ──
df["MAX_RISK_NUM"] = df[["FIRE_RISK_NUM","FLOOD_RISK_NUM",
                          "LANDSLIDE_RISK_NUM","HEATWAVE_RISK_NUM"]].max(axis=1)

risk_reverse = {0:"MINIMAL", 1:"LOW", 2:"MODERATE", 3:"HIGH", 4:"EXTREME"}
df["MAX_RISK"] = df["MAX_RISK_NUM"].map(risk_reverse)

print("Risk flags added ✅")
print("\nFIRE_RISK distribution:")
print(df["FIRE_RISK"].value_counts().to_string())
print("\nFLOOD_RISK distribution:")
print(df["FLOOD_RISK"].value_counts().to_string())
print("\nLANDSLIDE_RISK distribution:")
print(df["LANDSLIDE_RISK"].value_counts().to_string())
print("\nHEATWAVE_RISK distribution:")
print(df["HEATWAVE_RISK"].value_counts().to_string())

"""
Why normalize?
In Step 4 we feed data into a machine learning model.
ML models work best when all numbers are on the same scale.
A temperature of 25°C and humidity of 70% are very different
numbers — normalization makes them comparable.
"""

for param in PARAMS:
    col_min = df[param].min()
    col_max = df[param].max()
    if col_max > col_min:
        df[param + "_NORM"] = ((df[param] - col_min) / (col_max - col_min)).round(4)
    else:
        df[param + "_NORM"] = 0.0

print("Normalized columns added (suffix _NORM) ✅")

print("\n" + "=" * 60)
print("  STEP 2 — PREPROCESSING QUALITY REPORT")
print("=" * 60)
print(f"\n  Total records     : {len(df):,}")
print(f"  Total columns     : {df.shape[1]}")
print(f"  Date range        : {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"  Grid              : {df['LAT'].nunique()} lats × {df['LON'].nunique()} lons")
print(f"  Any NaN remaining : {df.isnull().any().any()}")

print(f"\n  Columns in final DataFrame:")
for col in df.columns:
    dtype = str(df[col].dtype)
    sample = str(df[col].iloc[0])[:20]
    print(f"    {col:<25} {dtype:<10} e.g. {sample}")

print(f"\n  Risk summary (% of records at each level):")
for risk_col in ["FIRE_RISK","FLOOD_RISK","LANDSLIDE_RISK","HEATWAVE_RISK"]:
    counts = df[risk_col].value_counts(normalize=True) * 100
    parts  = "  |  ".join(f"{k}: {counts.get(k,0):.1f}%" for k in
                          ["EXTREME","HIGH","MODERATE","LOW","MINIMAL"])
    print(f"    {risk_col:<18}: {parts}")


os.makedirs("preprocessed", exist_ok=True)

# ── CSV ──
csv_path = "preprocessed/uttarakhand_preprocessed.csv"
df.to_csv(csv_path, index=False)

# ── JSON (for dashboard later) ──
json_path = "preprocessed/uttarakhand_preprocessed.json"
df_json   = df.copy()
df_json["DATE"] = df_json["DATE"].astype(str)
df_json.to_json(json_path, orient="records", indent=2)

print(f"Saved:")
csv_size  = os.path.getsize(csv_path)  / (1024*1024)
json_size = os.path.getsize(json_path) / (1024*1024)
print(f"  {csv_path:<45} ({csv_size:.1f} MB)")
print(f"  {json_path:<45} ({json_size:.1f} MB)")

print(f"\n{'='*60}")
print(f"  STEP 2 COMPLETE ✅")
print(f"  Input  : 24,156 rows, 13 columns (raw weather)")
print(f"  Output : {len(df):,} rows, {df.shape[1]} columns (enriched + risk flags)")
print(f"{'='*60}")

Imports ready ✅
Loaded : 24,156 rows × 14 columns
Columns: ['LAT', 'LON', 'YEAR', 'MO', 'DY', 'DATE', 'RH2M', 'T2M', 'PRECTOTCORR', 'WS10M', 'T2M_MAX', 'T2M_MIN', 'GWETROOT', 'ALLSKY_SFC_SW_DWN']

First 3 rows:
    LAT     LON  YEAR  MO  DY       DATE   RH2M    T2M  PRECTOTCORR  WS10M  T2M_MAX  T2M_MIN  GWETROOT  ALLSKY_SFC_SW_DWN
0  28.0  77.500  2025   3   1 2025-03-01  62.47  21.16         2.91   3.26    29.33    14.75      0.36                NaN
1  28.0  78.125  2025   3   1 2025-03-01  60.60  21.08         1.56   3.36    29.22    14.78      0.36                NaN
2  28.0  78.750  2025   3   1 2025-03-01  61.07  20.94         3.51   3.20    28.97    14.99      0.36                NaN
Missing values BEFORE cleaning:
  RH2M                  : 18.2%
  T2M                   : 18.2%
  PRECTOTCORR           : 18.2%
  WS10M                 : 18.2%
  T2M_MAX               : 18.2%
  T2M_MIN               : 18.2%
  GWETROOT              : 18.2%
  ALLSKY_SFC_SW_DWN     : 75.8%

Dropped ALLS

In [ ]:
# One heavy rain day is fine. Seven consecutive days of rain = landslide danger.
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os
import warnings
warnings.filterwarnings("ignore")

os.makedirs("visualizations", exist_ok=True)
print("Imports ready")

Imports ready


In [ ]:
df = pd.read_csv("preprocessed/uttarakhand_preprocessed.csv", parse_dates=["DATE"])

# Risk numeric map
risk_map     = {"MINIMAL": 0, "LOW": 1, "MODERATE": 2, "HIGH": 3, "EXTREME": 4}
risk_reverse = {0: "MINIMAL", 1: "LOW", 2: "MODERATE", 3: "HIGH", 4: "EXTREME"}
RISK_COLORS  = ["#1a472a", "#2d6a4f", "#f4a261", "#e76f51", "#c1121f"]
RISK_LABELS  = ["MINIMAL", "LOW", "MODERATE", "HIGH", "EXTREME"]

print(f"Loaded : {len(df):,} rows")
print(f"Date   : {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Grid   : {df['LAT'].nunique()} lats × {df['LON'].nunique()} lons")

Loaded : 24,156 rows
Date   : 2025-03-01 → 2026-03-01
Grid   : 9 lats × 9 lons


In [ ]:
grid_avg = df.groupby(["LAT", "LON"]).agg(
    FIRE_RISK_NUM      = ("FIRE_RISK_NUM",      "mean"),
    FLOOD_RISK_NUM     = ("FLOOD_RISK_NUM",     "mean"),
    LANDSLIDE_RISK_NUM = ("LANDSLIDE_RISK_NUM", "mean"),
    HEATWAVE_RISK_NUM  = ("HEATWAVE_RISK_NUM",  "mean"),
    MAX_RISK_NUM       = ("MAX_RISK_NUM",        "mean"),
).reset_index().round(2)

fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Forest Fire Risk",
        "Flood Risk",
        "Landslide Risk",
        "Heat Wave Risk"
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)


In [ ]:
risk_cols = ["FIRE_RISK_NUM", "FLOOD_RISK_NUM", "LANDSLIDE_RISK_NUM", "HEATWAVE_RISK_NUM"]
positions = [(1,1), (1,2), (2,1), (2,2)]
color_scales = ["YlOrRd", "Blues", "RdPu", "YlOrBr"]

for risk_col, (row, col), cscale in zip(risk_cols, positions, color_scales):
    fig1.add_trace(
        go.Heatmap(
            x=grid_avg["LON"],
            y=grid_avg["LAT"],
            z=grid_avg[risk_col],
            colorscale=cscale,
            zmin=0, zmax=4,
            text=grid_avg[risk_col].apply(lambda x: f"{x:.2f}"),
            texttemplate="%{text}",
            showscale=True,
            colorbar=dict(
                tickvals=[0,1,2,3,4],
                ticktext=["Minimal","Low","Moderate","High","Extreme"],
                len=0.4,
                thickness=12,
            ),
            hoverongaps=False,
            hovertemplate="LAT: %{y}<br>LON: %{x}<br>Avg Risk: %{z:.2f}<extra></extra>"
        ),
        row=row, col=col
    )


In [ ]:
fig1.update_layout(
    title=dict(
        text="Uttarakhand — Annual Average Disaster Risk by Grid Cell (Mar 2025 – Mar 2026)",
        font=dict(size=16, color="#1a1a2e")
    ),
    height=700,
    paper_bgcolor="#f8f9fa",
    plot_bgcolor="#f8f9fa",
    font=dict(family="Arial", size=11)
)

fig1.update_xaxes(title_text="Longitude (°E)")
fig1.update_yaxes(title_text="Latitude (°N)")

fig1.write_html("visualizations/chart1_spatial_heatmap.html")
fig1.show()
print("Chart 1 saved→ visualizations/chart1_spatial_heatmap.html")

Chart 1 saved→ visualizations/chart1_spatial_heatmap.html


In [ ]:
monthly = df.groupby(["YEAR", "MO"]).agg(
    FIRE      = ("FIRE_RISK_NUM",      "mean"),
    FLOOD     = ("FLOOD_RISK_NUM",     "mean"),
    LANDSLIDE = ("LANDSLIDE_RISK_NUM", "mean"),
    HEATWAVE  = ("HEATWAVE_RISK_NUM",  "mean"),
).reset_index()

# Build a readable month label
import calendar
monthly["MONTH_LABEL"] = monthly.apply(
    lambda r: f"{calendar.month_abbr[int(r['MO'])]} {int(r['YEAR'])}", axis=1
)
monthly = monthly.sort_values(["YEAR", "MO"]).reset_index(drop=True)

fig2 = go.Figure()

In [ ]:
disaster_config = {
    "FIRE"      : ("#e63946", "Forest Fire",  "solid"),
    "FLOOD"     : ("#4895ef", "Flood",         "dot"),
    "LANDSLIDE" : ("#7209b7", "Landslide",    "dash"),
    "HEATWAVE"  : ("#f77f00", "Heat Wave",    "longdash"),
}

for col, (color, label, dash) in disaster_config.items():
    fig2.add_trace(go.Scatter(
        x=monthly["MONTH_LABEL"],
        y=monthly[col],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=7),
        hovertemplate=f"{label}<br>Month: %{{x}}<br>Avg Risk Score: %{{y:.2f}}<extra></extra>"
    ))

In [ ]:
season_bands = [
    ("Mar 2025", "May 2025", "rgba(255,165,0,0.08)", "Pre-Monsoon"),
    ("Jun 2025", "Sep 2025", "rgba(0,100,255,0.08)", "Monsoon"),
    ("Oct 2025", "Nov 2025", "rgba(100,200,100,0.08)", "Post-Monsoon"),
    ("Dec 2025", "Feb 2026", "rgba(150,150,255,0.08)", "Winter"),
]

fig2.update_layout(
    title=dict(
        text="📅 Monthly Disaster Risk Trends — Uttarakhand (Mar 2025 – Mar 2026)",
        font=dict(size=15)
    ),
    xaxis=dict(title="Month", tickangle=-30),
    yaxis=dict(
        title="Average Risk Score",
        tickvals=[0,1,2,3,4],
        ticktext=["Minimal","Low","Moderate","High","Extreme"],
        range=[-0.1, 4.2]
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=500,
    hovermode="x unified",
    paper_bgcolor="#ffffff",
    plot_bgcolor="#fafafa",
    font=dict(family="Arial", size=12),
    shapes=[
        dict(type="rect", xref="paper", yref="paper",
             x0=0, x1=0.23, y0=0, y1=1,
             fillcolor="rgba(255,165,0,0.06)", line_width=0),
        dict(type="rect", xref="paper", yref="paper",
             x0=0.23, x1=0.54, y0=0, y1=1,
             fillcolor="rgba(0,100,255,0.06)", line_width=0),
        dict(type="rect", xref="paper", yref="paper",
             x0=0.54, x1=0.69, y0=0, y1=1,
             fillcolor="rgba(100,200,100,0.06)", line_width=0),
        dict(type="rect", xref="paper", yref="paper",
             x0=0.69, x1=1.0, y0=0, y1=1,
             fillcolor="rgba(150,150,255,0.06)", line_width=0),
    ]
)


In [ ]:
fig2.write_html("visualizations/chart2_monthly_timeline.html")
fig2.show()
print("Chart 2 saved → visualizations/chart2_monthly_timeline.html")

Chart 2 saved → visualizations/chart2_monthly_timeline.html


In [ ]:
season_order  = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]
season_colors = ["#4cc9f0", "#f4a261", "#4361ee", "#7cb518"]

seasonal = df.groupby("SEASON").agg(
    FIRE      = ("FIRE_RISK_NUM",      "mean"),
    FLOOD     = ("FLOOD_RISK_NUM",     "mean"),
    LANDSLIDE = ("LANDSLIDE_RISK_NUM", "mean"),
    HEATWAVE  = ("HEATWAVE_RISK_NUM",  "mean"),
).reindex(season_order).reset_index()

In [ ]:
fig3 = go.Figure()

disaster_labels = {
    "FIRE"      : "Forest Fire",
    "FLOOD"     : "Flood",
    "LANDSLIDE" : "Landslide",
    "HEATWAVE"  : "Heat Wave",
}
bar_colors = ["#e63946", "#4895ef", "#7209b7", "#f77f00"]

for (col, label), color in zip(disaster_labels.items(), bar_colors):
    fig3.add_trace(go.Bar(
        name=label,
        x=seasonal["SEASON"],
        y=seasonal[col],
        marker_color=color,
        text=seasonal[col].round(2),
        textposition="outside",
        hovertemplate=f"{label}<br>Season: %{{x}}<br>Avg Risk: %{{y:.2f}}<extra></extra>"
    ))

In [ ]:
fig3.update_layout(
    title=dict(
        text="🌿 Seasonal Risk Breakdown — Which Season is Most Dangerous?",
        font=dict(size=15)
    ),
    barmode="group",
    xaxis=dict(title="Season", categoryorder="array", categoryarray=season_order),
    yaxis=dict(
        title="Average Risk Score",
        tickvals=[0,1,2,3,4],
        ticktext=["Minimal","Low","Moderate","High","Extreme"],
        range=[0, 4.5]
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=480,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#fafafa",
    font=dict(family="Arial", size=12),
)

fig3.write_html("visualizations/chart3_seasonal_breakdown.html")
fig3.show()
print("Chart 3 saved → visualizations/chart3_seasonal_breakdown.html")



Chart 3 saved → visualizations/chart3_seasonal_breakdown.html


In [ ]:
corr_cols = [
    "RH2M", "T2M", "PRECTOTCORR", "WS10M", "T2M_MAX", "T2M_MIN", "GWETROOT",
    "FIRE_RISK_NUM", "FLOOD_RISK_NUM", "LANDSLIDE_RISK_NUM", "HEATWAVE_RISK_NUM"
]
corr_labels = [
    "Humidity", "Temperature", "Rainfall", "Wind Speed",
    "Max Temp", "Min Temp", "Soil Wetness",
    "Fire Risk", "Flood Risk", "Landslide Risk", "Heatwave Risk"
]

In [ ]:
corr_matrix = df[corr_cols].corr().round(2)

fig4 = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_labels,
    y=corr_labels,
    colorscale="RdBu_r",
    zmin=-1, zmax=1,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont=dict(size=10),
    hoverongaps=False,
    hovertemplate="X: %{x}<br>Y: %{y}<br>Correlation: %{z}<extra></extra>"
))

In [ ]:
fig4.update_layout(
    title=dict(
        text="🔗 Parameter Correlation Matrix — How Weather Drives Disaster Risk",
        font=dict(size=15)
    ),
    height=580,
    width=780,
    paper_bgcolor="#ffffff",
    font=dict(family="Arial", size=11),
    xaxis=dict(tickangle=-35),
)

fig4.write_html("visualizations/chart4_correlation.html")
fig4.show()
print("Chart 4 saved → visualizations/chart4_correlation.html")


Chart 4 saved → visualizations/chart4_correlation.html


In [ ]:
daily_avg = df.groupby("DATE").agg(
    MAX_RISK   = ("MAX_RISK_NUM",        "mean"),
    FIRE       = ("FIRE_RISK_NUM",       "mean"),
    FLOOD      = ("FLOOD_RISK_NUM",      "mean"),
    LANDSLIDE  = ("LANDSLIDE_RISK_NUM",  "mean"),
    HEATWAVE   = ("HEATWAVE_RISK_NUM",   "mean"),
).reset_index()

daily_avg["WEEK_NUM"]   = daily_avg["DATE"].dt.isocalendar().week.astype(int)
daily_avg["DAY_OF_WK"]  = daily_avg["DATE"].dt.dayofweek   # 0=Mon
daily_avg["MONTH_NAME"] = daily_avg["DATE"].dt.strftime("%b %Y")


In [ ]:
fig5 = go.Figure(go.Heatmap(
    x=daily_avg["DATE"],
    y=["Risk"] * len(daily_avg),
    z=daily_avg["MAX_RISK"],
    colorscale=[
        [0.0,  "#1a472a"],
        [0.25, "#52b788"],
        [0.5,  "#f4a261"],
        [0.75, "#e76f51"],
        [1.0,  "#c1121f"],
    ],
    zmin=0, zmax=4,
    colorbar=dict(
        tickvals=[0,1,2,3,4],
        ticktext=["Minimal","Low","Moderate","High","Extreme"],
        thickness=15,
    ),
    hovertemplate=(
        "Date: %{x|%d %b %Y}<br>"
        "Max Risk Score: %{z:.2f}<extra></extra>"
    )
))


In [ ]:
fig5.update_layout(
    title=dict(
        text="📆 Daily Overall Risk Calendar — Uttarakhand (Mar 2025 – Mar 2026)",
        font=dict(size=15)
    ),
    height=220,
    xaxis=dict(title="", dtick="M1", tickformat="%b %Y", tickangle=-30),
    yaxis=dict(showticklabels=False),
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    font=dict(family="Arial", size=12),
)

fig5.write_html("visualizations/chart5_risk_calendar.html")
fig5.show()
print("Chart 5 saved → visualizations/chart5_risk_calendar.html")

Chart 5 saved → visualizations/chart5_risk_calendar.html


In [ ]:
top_days = (
    df.groupby("DATE").agg(
        FIRE      = ("FIRE_RISK_NUM",      "max"),
        FLOOD     = ("FLOOD_RISK_NUM",     "max"),
        LANDSLIDE = ("LANDSLIDE_RISK_NUM", "max"),
        HEATWAVE  = ("HEATWAVE_RISK_NUM",  "max"),
        MAX_RISK  = ("MAX_RISK_NUM",        "max"),
    )
    .reset_index()
    .sort_values("MAX_RISK", ascending=False)
    .head(10)
)

top_days["DATE_STR"] = top_days["DATE"].dt.strftime("%d %b %Y")

fig6 = go.Figure()

cols_config = [
    ("FIRE",      "#e63946", "Fire"),
    ("FLOOD",     "#4895ef", "Flood"),
    ("LANDSLIDE", "#7209b7", "Landslide"),
    ("HEATWAVE",  "#f77f00", "Heat Wave"),
]

for col, color, label in cols_config:
    fig6.add_trace(go.Bar(
        name=label,
        x=top_days["DATE_STR"],
        y=top_days[col],
        marker_color=color,
        hovertemplate=f"{label}<br>Date: %{{x}}<br>Risk: %{{y}}<extra></extra>"
    ))

fig6.update_layout(
    title=dict(
        text="Top 10 Highest Risk Days in Uttarakhand (All Grid Cells)",
        font=dict(size=15)
    ),
    barmode="group",
    xaxis=dict(title="Date", tickangle=-30),
    yaxis=dict(
        title="Max Risk Score",
        tickvals=[0,1,2,3,4],
        ticktext=["Minimal","Low","Moderate","High","Extreme"],
        range=[0, 4.5]
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=480,
    paper_bgcolor="#ffffff",
    plot_bgcolor="#fafafa",
    font=dict(family="Arial", size=12),
)

fig6.write_html("visualizations/chart6_top_risk_days.html")
fig6.show()
print("Chart 6 saved → visualizations/chart6_top_risk_days.html")



Chart 6 saved → visualizations/chart6_top_risk_days.html


In [ ]:
print("\n" + "=" * 60)
print("  STEP 3 — VISUALIZATIONS COMPLETE ✅")
print("=" * 60)
print("\nFiles saved in ./visualizations/:")
for f in sorted(os.listdir("visualizations")):
    size = os.path.getsize(f"visualizations/{f}") / 1024
    print(f"  {f:<45} ({size:.0f} KB)")


  STEP 3 — VISUALIZATIONS COMPLETE ✅

Files saved in ./visualizations/:
  chart1_spatial_heatmap.html                   (4469 KB)
  chart2_monthly_timeline.html                  (4463 KB)
  chart3_seasonal_breakdown.html                (4461 KB)
  chart4_correlation.html                       (4462 KB)
  chart5_risk_calendar.html                     (4477 KB)
  chart6_top_risk_days.html                     (4461 KB)
  uttarakhand_risk_map.html                     (1418 KB)


In [ ]:
'''Key Findings from our data:
─────────────────────────────────────────────
  Landslide : Most persistent risk (50% records not MINIMAL)
  Fire      : Peaks in Pre-Monsoon (Mar–May)
  Flood     : Peaks in Monsoon (Jun–Sep)
  Heat Wave : Peaks in Pre-Monsoon (Apr–Jun)'''

'Key Findings from our data:\n─────────────────────────────────────────────\n  Landslide : Most persistent risk (50% records not MINIMAL)\n  Fire      : Peaks in Pre-Monsoon (Mar–May)\n  Flood     : Peaks in Monsoon (Jun–Sep)\n  Heat Wave : Peaks in Pre-Monsoon (Apr–Jun)'

In [ ]:
# !pip install folium --quiet

import folium
from folium.plugins import HeatMap
import pandas as pd
import numpy as np
import json
import os

print("Folium ready ✅")


df = pd.read_csv("preprocessed/uttarakhand_preprocessed.csv", parse_dates=["DATE"])
print(f"Loaded {len(df):,} rows ✅")


grid = df.groupby(["LAT", "LON"]).agg(
    FIRE_RISK_AVG      = ("FIRE_RISK_NUM",      "mean"),
    FLOOD_RISK_AVG     = ("FLOOD_RISK_NUM",     "mean"),
    LANDSLIDE_RISK_AVG = ("LANDSLIDE_RISK_NUM", "mean"),
    HEATWAVE_RISK_AVG  = ("HEATWAVE_RISK_NUM",  "mean"),
    MAX_RISK_AVG       = ("MAX_RISK_NUM",        "mean"),

    FIRE_HIGH_DAYS     = ("FIRE_RISK_NUM",      lambda x: (x >= 3).sum()),
    FLOOD_HIGH_DAYS    = ("FLOOD_RISK_NUM",     lambda x: (x >= 3).sum()),
    LANDSLIDE_HIGH_DAYS= ("LANDSLIDE_RISK_NUM", lambda x: (x >= 3).sum()),
    HEATWAVE_HIGH_DAYS = ("HEATWAVE_RISK_NUM",  lambda x: (x >= 3).sum()),

    PRECIP_MEAN        = ("PRECTOTCORR",        "mean"),
    TEMP_MEAN          = ("T2M",                "mean"),
    RH2M_MEAN          = ("RH2M",              "mean"),
    WIND_MEAN          = ("WS10M",             "mean"),
    SOIL_MEAN          = ("GWETROOT",          "mean"),
).reset_index().round(3)

print(f"Grid cells: {len(grid)}")
print(grid.head())


def risk_to_color(score):
    """Convert 0–4 risk score to hex colour."""
    if score < 0.5:   return "#2d6a4f"   # dark green  = MINIMAL
    elif score < 1.5: return "#95d5b2"   # light green = LOW
    elif score < 2.5: return "#f4a261"   # orange      = MODERATE
    elif score < 3.5: return "#e76f51"   # red-orange  = HIGH
    else:             return "#c1121f"   # dark red    = EXTREME

def risk_to_label(score):
    if score < 0.5:   return "MINIMAL"
    elif score < 1.5: return "LOW"
    elif score < 2.5: return "MODERATE"
    elif score < 3.5: return "HIGH"
    else:             return "EXTREME"

def score_to_bar(score, max_score=4):
    """Return a mini HTML progress bar."""
    pct   = int((score / max_score) * 100)
    color = risk_to_color(score)
    return (f'<div style="background:#eee;border-radius:3px;height:8px;width:100%">'
            f'<div style="background:{color};width:{pct}%;height:8px;border-radius:3px"></div></div>')


# Centre of Uttarakhand
UK_CENTER = [30.0, 79.0]
CELL_SIZE  = 0.3125   # half of 0.625 degree lon step

m = folium.Map(
    location=UK_CENTER,
    zoom_start=7,
    tiles=None,
)

# ── Base tile layers ──
folium.TileLayer(
    "CartoDB positron",
    name="Light Map",
    control=True
).add_to(m)

folium.TileLayer(
    "CartoDB dark_matter",
    name="Dark Map",
    control=True
).add_to(m)

folium.TileLayer(
    "OpenStreetMap",
    name="Street Map",
    control=True
).add_to(m)


def make_layer(df_grid, risk_col, layer_name, emoji):
    """Creates a FeatureGroup layer with coloured rectangles."""
    layer = folium.FeatureGroup(name=f"{emoji} {layer_name}", show=True)

    for _, row in df_grid.iterrows():
        lat  = row["LAT"]
        lon  = row["LON"]
        score= row[risk_col]
        color= risk_to_color(score)
        label= risk_to_label(score)

        # Rectangle bounds (half cell size in each direction)
        bounds = [
            [lat - 0.25, lon - 0.3125],
            [lat + 0.25, lon + 0.3125]
        ]

        # ── Popup with full stats ──
        popup_html = f"""
        <div style="font-family:'Segoe UI',sans-serif;min-width:220px;padding:4px">
          <div style="font-size:14px;font-weight:700;margin-bottom:8px;
                      color:#1a1a2e;border-bottom:2px solid {color};padding-bottom:4px">
            📍 Grid Cell: {lat}°N, {lon}°E
          </div>

          <div style="margin-bottom:10px">
            <div style="font-size:11px;color:#666;margin-bottom:2px">
              {emoji} {layer_name} Risk
            </div>
            <div style="font-size:16px;font-weight:700;color:{color}">{label}</div>
            <div style="margin-top:4px">{score_to_bar(score)}</div>
            <div style="font-size:11px;color:#888;margin-top:2px">Score: {score:.2f} / 4.0</div>
          </div>

          <table style="width:100%;font-size:11px;border-collapse:collapse">
            <tr style="background:#f5f5f5">
              <td style="padding:3px 6px;color:#444">🔥 Fire Risk</td>
              <td style="padding:3px 6px;font-weight:600;color:{risk_to_color(row['FIRE_RISK_AVG'])}">
                {risk_to_label(row['FIRE_RISK_AVG'])} ({row['FIRE_RISK_AVG']:.2f})
              </td>
            </tr>
            <tr>
              <td style="padding:3px 6px;color:#444">🌊 Flood Risk</td>
              <td style="padding:3px 6px;font-weight:600;color:{risk_to_color(row['FLOOD_RISK_AVG'])}">
                {risk_to_label(row['FLOOD_RISK_AVG'])} ({row['FLOOD_RISK_AVG']:.2f})
              </td>
            </tr>
            <tr style="background:#f5f5f5">
              <td style="padding:3px 6px;color:#444">⛰️ Landslide</td>
              <td style="padding:3px 6px;font-weight:600;color:{risk_to_color(row['LANDSLIDE_RISK_AVG'])}">
                {risk_to_label(row['LANDSLIDE_RISK_AVG'])} ({row['LANDSLIDE_RISK_AVG']:.2f})
              </td>
            </tr>
            <tr>
              <td style="padding:3px 6px;color:#444">🌡️ Heat Wave</td>
              <td style="padding:3px 6px;font-weight:600;color:{risk_to_color(row['HEATWAVE_RISK_AVG'])}">
                {risk_to_label(row['HEATWAVE_RISK_AVG'])} ({row['HEATWAVE_RISK_AVG']:.2f})
              </td>
            </tr>
          </table>

          <div style="margin-top:10px;padding-top:6px;border-top:1px solid #eee">
            <div style="font-size:11px;color:#555;font-weight:600;margin-bottom:4px">
              📊 Annual Weather Summary
            </div>
            <div style="font-size:11px;color:#666">
              🌡️ Avg Temp: <b>{row['TEMP_MEAN']:.1f}°C</b> &nbsp;
              💧 Humidity: <b>{row['RH2M_MEAN']:.1f}%</b><br>
              🌧️ Avg Rain: <b>{row['PRECIP_MEAN']:.1f} mm/day</b> &nbsp;
              💨 Wind: <b>{row['WIND_MEAN']:.1f} m/s</b><br>
              🌱 Soil Wet: <b>{row['SOIL_MEAN']:.2f}</b>
            </div>
          </div>

          <div style="margin-top:8px;padding-top:6px;border-top:1px solid #eee">
            <div style="font-size:11px;color:#555;font-weight:600;margin-bottom:4px">
              🚨 High Risk Days (per year)
            </div>
            <div style="font-size:11px;color:#666">
              Fire: <b>{int(row['FIRE_HIGH_DAYS'])} days</b> &nbsp;
              Flood: <b>{int(row['FLOOD_HIGH_DAYS'])} days</b><br>
              Landslide: <b>{int(row['LANDSLIDE_HIGH_DAYS'])} days</b> &nbsp;
              Heat: <b>{int(row['HEATWAVE_HIGH_DAYS'])} days</b>
            </div>
          </div>
        </div>
        """

        folium.Rectangle(
            bounds=bounds,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.55,
            weight=1,
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=f"{emoji} {layer_name}: {label} ({score:.2f})",
        ).add_to(layer)

    return layer

# Build 4 layers + 1 overall
layers = {
    "FIRE_RISK_AVG"      : ("Forest Fire",  "🔥"),
    "FLOOD_RISK_AVG"     : ("Flood",        "🌊"),
    "LANDSLIDE_RISK_AVG" : ("Landslide",    "⛰️"),
    "HEATWAVE_RISK_AVG"  : ("Heat Wave",    "🌡️"),
    "MAX_RISK_AVG"       : ("Overall Risk", "⚠️"),
}

for risk_col, (name, emoji) in layers.items():
    layer = make_layer(grid, risk_col, name, emoji)
    layer.add_to(m)

print("Risk layers added ✅")


districts = [
    ("Dehradun",     30.316,  78.032, "State Capital"),
    ("Haridwar",     29.946,  78.162, "Holy City / Flood Prone"),
    ("Rishikesh",    30.087,  78.268, "Gateway to Himalayas"),
    ("Nainital",     29.381,  79.463, "Lake District"),
    ("Almora",       29.597,  79.646, "Cultural Hub"),
    ("Mussoorie",    30.454,  78.065, "Hill Station"),
    ("Pithoragarh",  29.582,  80.218, "Border District"),
    ("Uttarkashi",   30.726,  78.440, "High Altitude"),
    ("Chamoli",      30.401,  79.321, "Kedarnath Region"),
    ("Tehri",        30.378,  78.480, "Dam / Reservoir"),
    ("Rudraprayag",  30.284,  78.981, "Confluence District"),
    ("Bageshwar",    29.838,  79.770, "Mountain District"),
    ("Champawat",    29.335,  80.091, "Eastern Border"),
    ("Pauri",        29.771,  78.780, "Garhwal Division"),
]

district_layer = folium.FeatureGroup(name="📍 Districts & Cities", show=True)

for name, lat, lon, desc in districts:
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(
            f"""<div style="font-family:'Segoe UI',sans-serif;padding:4px">
                <b style="font-size:13px">{name}</b><br>
                <span style="color:#666;font-size:11px">{desc}</span><br>
                <span style="color:#888;font-size:10px">📍 {lat}°N, {lon}°E</span>
            </div>""",
            max_width=180
        ),
        tooltip=f"📍 {name}",
        icon=folium.Icon(
            color="darkblue",
            icon="info-sign",
            prefix="glyphicon"
        )
    ).add_to(district_layer)

district_layer.add_to(m)
print("District markers added ✅")


legend_html = """
<div style="
    position: fixed;
    bottom: 30px; right: 15px;
    z-index: 9999;
    background: white;
    border: 2px solid #ccc;
    border-radius: 10px;
    padding: 14px 18px;
    font-family: 'Segoe UI', sans-serif;
    box-shadow: 0 4px 15px rgba(0,0,0,0.15);
    min-width: 180px;
">
    <div style="font-size:13px;font-weight:700;margin-bottom:10px;
                color:#1a1a2e;border-bottom:1px solid #eee;padding-bottom:6px">
        ⚠️ Risk Level Legend
    </div>
    <div style="display:flex;align-items:center;margin-bottom:6px">
        <div style="width:20px;height:14px;background:#c1121f;border-radius:3px;margin-right:8px"></div>
        <span style="font-size:12px;color:#333">EXTREME</span>
    </div>
    <div style="display:flex;align-items:center;margin-bottom:6px">
        <div style="width:20px;height:14px;background:#e76f51;border-radius:3px;margin-right:8px"></div>
        <span style="font-size:12px;color:#333">HIGH</span>
    </div>
    <div style="display:flex;align-items:center;margin-bottom:6px">
        <div style="width:20px;height:14px;background:#f4a261;border-radius:3px;margin-right:8px"></div>
        <span style="font-size:12px;color:#333">MODERATE</span>
    </div>
    <div style="display:flex;align-items:center;margin-bottom:6px">
        <div style="width:20px;height:14px;background:#95d5b2;border-radius:3px;margin-right:8px"></div>
        <span style="font-size:12px;color:#333">LOW</span>
    </div>
    <div style="display:flex;align-items:center;margin-bottom:10px">
        <div style="width:20px;height:14px;background:#2d6a4f;border-radius:3px;margin-right:8px"></div>
        <span style="font-size:12px;color:#333">MINIMAL</span>
    </div>
    <div style="font-size:10px;color:#888;border-top:1px solid #eee;padding-top:6px">
        Annual average (Mar 2025 – Mar 2026)<br>
        Click any cell for full details
    </div>
</div>
"""

title_html = """
<div style="
    position: fixed;
    top: 15px; left: 50%; transform: translateX(-50%);
    z-index: 9999;
    background: rgba(26,26,46,0.92);
    color: white;
    padding: 10px 24px;
    border-radius: 30px;
    font-family: 'Segoe UI', sans-serif;
    font-size: 14px;
    font-weight: 600;
    letter-spacing: 0.5px;
    box-shadow: 0 4px 20px rgba(0,0,0,0.3);
    pointer-events: none;
">
    🏔️ Uttarakhand Disaster Risk Map &nbsp;|&nbsp;
    <span style="color:#00d9a3">NASA POWER MERRA-2</span> &nbsp;|&nbsp;
    <span style="color:#f4a261">Mar 2025 – Mar 2026</span>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))
m.get_root().html.add_child(folium.Element(title_html))


folium.LayerControl(
    position="topright",
    collapsed=False
).add_to(m)

os.makedirs("visualizations", exist_ok=True)
map_path = "visualizations/uttarakhand_risk_map.html"
m.save(map_path)

size_kb = os.path.getsize(map_path) / 1024
print(f"\n{'='*55}")
print(f"  MAP SAVED ✅")
print(f"  File : {map_path}  ({size_kb:.0f} KB)")
print(f"{'='*55}")


Folium ready ✅
Loaded 24,156 rows ✅
Grid cells: 66
    LAT     LON  FIRE_RISK_AVG  FLOOD_RISK_AVG  LANDSLIDE_RISK_AVG  \
0  28.0  77.500          0.699           0.150               0.421   
1  28.0  78.125          0.653           0.145               0.454   
2  28.0  78.750          0.626           0.191               0.530   
3  28.0  79.375          0.623           0.186               0.634   
4  28.0  80.000          0.631           0.205               0.658   

   HEATWAVE_RISK_AVG  MAX_RISK_AVG  FIRE_HIGH_DAYS  FLOOD_HIGH_DAYS  \
0              1.109         1.544              45                8   
1              1.104         1.516              42                6   
2              1.104         1.560              40                8   
3              1.112         1.631              38                9   
4              1.101         1.631              37                8   

   LANDSLIDE_HIGH_DAYS  HEATWAVE_HIGH_DAYS  PRECIP_MEAN  TEMP_MEAN  RH2M_MEAN  \
0                   

In [ ]:
import os

files_to_check = [
    "preprocessed/uttarakhand_preprocessed.csv",
    "ml_results/uttarakhand_ml_predictions.csv",
    "alerts/all_alerts.json",
    "alerts/latest_snapshot.json",
    "dashboard/uttarakhand_dashboard.html"
]

print("FILE STATUS CHECK")
print("="*55)
for f in files_to_check:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024 if exists else 0
    status = f"✅  {size:.0f} KB" if exists else "❌  MISSING"
    print(f"  {status:<15}  {f}")

FILE STATUS CHECK
  ✅  6145 KB       preprocessed/uttarakhand_preprocessed.csv
  ❌  MISSING       ml_results/uttarakhand_ml_predictions.csv
  ❌  MISSING       alerts/all_alerts.json
  ❌  MISSING       alerts/latest_snapshot.json
  ❌  MISSING       dashboard/uttarakhand_dashboard.html


In [ ]:
import pandas as pd
import numpy as np
import json, os, warnings, pickle
warnings.filterwarnings("ignore")
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("preprocessed/uttarakhand_preprocessed.csv", parse_dates=["DATE"])
print(f"df loaded: {df.shape}")

df loaded: (24156, 46)


In [ ]:
FEATURES = ["RH2M","T2M","PRECTOTCORR","WS10M","T2M_MAX","T2M_MIN","GWETROOT",
            "RH2M_7D","PRECIP_7D","WIND_7D","SOIL_7D","PRECIP_3D_CUM",
            "TEMP_ANOMALY","RH2M_ANOMALY","PRECIP_ANOMALY","LAT","LON","MO","DAY_OF_YEAR"]
TARGETS   = {"FIRE":"FIRE_RISK","FLOOD":"FLOOD_RISK",
             "LANDSLIDE":"LANDSLIDE_RISK","HEATWAVE":"HEATWAVE_RISK"}
RISK_ORDER = ["MINIMAL","LOW","MODERATE","HIGH","EXTREME"]
df_ml = df[FEATURES + list(TARGETS.values())].dropna().copy()
print(f"df_ml: {df_ml.shape} ✅")

df_ml: (23760, 23) ✅


In [ ]:
#THE ALERT SYSTEM
#🟢 GREEN  (score 0–1.4)  → No Alert       → Monitor conditions
# 🟡 YELLOW (score 1.5–2.4) → Watch          → Stay alert, prepare
# 🟠 ORANGE (score 2.5–3.4) → Warning        → Avoid risk zones
# 🔴 RED    (score 3.5–4.0) → Emergency      → Evacuate immediately

In [ ]:
import pandas as pd
import numpy as np
import json, os, warnings, pickle
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

os.makedirs("ml_models", exist_ok=True)
os.makedirs("ml_results", exist_ok=True)

# ── Load data ──
df = pd.read_csv("preprocessed/uttarakhand_preprocessed.csv", parse_dates=["DATE"])
print(f"Loaded {len(df):,} rows ✅")

FEATURES = ["RH2M","T2M","PRECTOTCORR","WS10M","T2M_MAX","T2M_MIN","GWETROOT",
            "RH2M_7D","PRECIP_7D","WIND_7D","SOIL_7D","PRECIP_3D_CUM",
            "TEMP_ANOMALY","RH2M_ANOMALY","PRECIP_ANOMALY","LAT","LON","MO","DAY_OF_YEAR"]

TARGETS   = {"FIRE":"FIRE_RISK","FLOOD":"FLOOD_RISK","LANDSLIDE":"LANDSLIDE_RISK","HEATWAVE":"HEATWAVE_RISK"}
RISK_ORDER= ["MINIMAL","LOW","MODERATE","HIGH","EXTREME"]
RISK_NUM  = {"FIRE":"FIRE_RISK_NUM","FLOOD":"FLOOD_RISK_NUM","LANDSLIDE":"LANDSLIDE_RISK_NUM","HEATWAVE":"HEATWAVE_RISK_NUM"}

df_ml = df[FEATURES + list(TARGETS.values()) + list(RISK_NUM.values())].dropna().copy()
X     = df_ml[FEATURES]
print(f"Training rows: {len(df_ml):,}")

# ── Train models ──
models, encoders, importances = {}, {}, {}
for disaster, target_col in TARGETS.items():
    print(f"\nTraining {disaster}...")
    le = LabelEncoder(); le.fit(RISK_ORDER)
    y  = le.transform(df_ml[target_col])
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
    rf = RandomForestClassifier(n_estimators=150,max_depth=12,min_samples_leaf=5,
                                 class_weight="balanced",random_state=42,n_jobs=-1)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    print(f"  Accuracy: {acc*100:.1f}%")
    pickle.dump(rf, open(f"ml_models/{disaster}_model.pkl","wb"))
    models[disaster]      = rf
    encoders[disaster]    = le
    importances[disaster] = dict(zip(FEATURES, rf.feature_importances_.round(4)))

print("\nALL MODELS TRAINED ✅")

# ── Generate predictions ──
df_pred = df_ml.copy()
for disaster, model in models.items():
    le   = encoders[disaster]
    pred = model.predict(X)
    prob = model.predict_proba(X)
    df_pred[f"ML_{disaster}_RISK"]  = le.inverse_transform(pred)
    df_pred[f"ML_{disaster}_SCORE"] = pred
    df_pred[f"ML_{disaster}_CONF"]  = prob.max(axis=1).round(3)

# ── Combined score (40% rules + 60% ML) ──
def score_to_label(s):
    if s<0.5: return "MINIMAL"
    elif s<1.5: return "LOW"
    elif s<2.5: return "MODERATE"
    elif s<3.5: return "HIGH"
    else: return "EXTREME"

for disaster in TARGETS:
    rule_col = RISK_NUM[disaster]
    ml_col   = f"ML_{disaster}_SCORE"
    df_pred[f"FINAL_{disaster}_SCORE"] = (0.4*df_pred[rule_col].values + 0.6*df_pred[ml_col].values).round(3)
    df_pred[f"FINAL_{disaster}_RISK"]  = df_pred[f"FINAL_{disaster}_SCORE"].apply(score_to_label)

df_pred["FINAL_MAX_SCORE"] = df_pred[["FINAL_FIRE_SCORE","FINAL_FLOOD_SCORE",
                                       "FINAL_LANDSLIDE_SCORE","FINAL_HEATWAVE_SCORE"]].max(axis=1).round(3)
df_pred["FINAL_MAX_RISK"]  = df_pred["FINAL_MAX_SCORE"].apply(score_to_label)

# ── Save ──
df_pred.to_csv("ml_results/uttarakhand_ml_predictions.csv", index=False)
json.dump(importances, open("ml_results/feature_importance.json","w"), indent=2)

print(f"\nFinal risk distribution:")
print(df_pred["FINAL_MAX_RISK"].value_counts())
print(f"\n{'='*50}")
print(f"  STEP 4 COMPLETE ✅")
print(f"  Saved: ml_results/uttarakhand_ml_predictions.csv")
print(f"{'='*50}")

Loaded 24,156 rows ✅
Training rows: 23,760

Training FIRE...
  Accuracy: 99.4%

Training FLOOD...
  Accuracy: 99.1%

Training LANDSLIDE...
  Accuracy: 98.8%

Training HEATWAVE...
  Accuracy: 99.3%

ALL MODELS TRAINED ✅

Final risk distribution:
FINAL_MAX_RISK
MODERATE    19790
HIGH         3937
EXTREME        33
Name: count, dtype: int64

  STEP 4 COMPLETE ✅
  Saved: ml_results/uttarakhand_ml_predictions.csv


In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

os.makedirs("alerts", exist_ok=True)
print("Imports ready ✅")

# CELL 2 — LOAD ML PREDICTIONS
import pandas as pd
import numpy as np

df = pd.read_csv("ml_results/uttarakhand_ml_predictions.csv")
df["YEAR"] = df["MO"].apply(lambda m: 2025 if m >= 3 else 2026)
df["DATE"] = pd.to_datetime(
    df["YEAR"].astype(str) + "-" + df["DAY_OF_YEAR"].astype(str),
    format="%Y-%j"
)

print(f"Loaded : {len(df):,} rows")
print(f"Date range: {df['DATE'].min().date()} → {df['DATE'].max().date()}")
print(f"Sample dates: {df['DATE'].head(3).tolist()}")

Imports ready ✅
Loaded : 23,760 rows
Date range: 2025-03-01 → 2026-02-28
Sample dates: [Timestamp('2025-03-07 00:00:00'), Timestamp('2025-03-08 00:00:00'), Timestamp('2025-03-09 00:00:00')]


In [ ]:
# CELL 3 — DISTRICT MAPPING
# Map each grid cell (latitude/longitude) to nearest Uttarakhand district
# Uttarakhand districts with approximate centre coordinates
DISTRICTS = {
    "Dehradun"    : (30.316, 78.032),
    "Haridwar"    : (29.946, 78.162),
    "Pauri"       : (29.771, 78.780),
    "Tehri"       : (30.378, 78.480),
    "Uttarkashi"  : (30.726, 78.440),
    "Chamoli"     : (30.401, 79.321),
    "Rudraprayag" : (30.284, 78.981),
    "Nainital"    : (29.381, 79.463),
    "Almora"      : (29.597, 79.646),
    "Bageshwar"   : (29.838, 79.770),
    "Pithoragarh" : (29.582, 80.218),
    "Champawat"   : (29.335, 80.091),
}

def nearest_district(lat, lon):
    """Find nearest district to a grid cell."""
    min_dist = float("inf")
    nearest  = "Unknown"
    for district, (dlat, dlon) in DISTRICTS.items():
        dist = ((lat - dlat)**2 + (lon - dlon)**2) ** 0.5
        if dist < min_dist:
            min_dist = dist
            nearest  = district
    return nearest

df["DISTRICT"] = df.apply(
    lambda r: nearest_district(r["LAT"], r["LON"]), axis=1
)

print("District mapping done ✅")
print(df["DISTRICT"].value_counts())

District mapping done ✅
DISTRICT
Chamoli        5040
Uttarkashi     3600
Nainital       2880
Champawat      2880
Pauri          2520
Haridwar       2520
Pithoragarh    1800
Dehradun       1440
Tehri           720
Bageshwar       360
Name: count, dtype: int64


In [ ]:
# CELL 4 — ALERT THRESHOLDS & TRIGGER LOGIC

ALERT_LEVELS = {
    "GREEN"    : {"min": 0.0, "max": 1.4, "emoji": "🟢", "label": "No Alert",   "action": "Monitor conditions"},
    "YELLOW"   : {"min": 1.5, "max": 2.4, "emoji": "🟡", "label": "Watch",      "action": "Stay alert, prepare"},
    "ORANGE"   : {"min": 2.5, "max": 3.4, "emoji": "🟠", "label": "Warning",    "action": "Take precautions, avoid risk zones"},
    "RED"      : {"min": 3.5, "max": 4.0, "emoji": "🔴", "label": "Emergency",  "action": "Evacuate immediately, emergency response"},
}

DISASTER_NAMES = {
    "FIRE"      : "Forest Fire",
    "FLOOD"     : "Flood",
    "LANDSLIDE" : "Landslide",
    "HEATWAVE"  : "Heat Wave",
}

def get_alert_level(score):
    if score >= 3.5:  return "RED"
    elif score >= 2.5: return "ORANGE"
    elif score >= 1.5: return "YELLOW"
    else:              return "GREEN"

# Add alert level columns
for disaster in ["FIRE", "FLOOD", "LANDSLIDE", "HEATWAVE"]:
    score_col = f"FINAL_{disaster}_SCORE"
    df[f"ALERT_{disaster}"] = df[score_col].apply(get_alert_level)

df["ALERT_MAX"] = df["FINAL_MAX_SCORE"].apply(get_alert_level)

print("\nAlert level distribution (overall):")
print(df["ALERT_MAX"].value_counts())


Alert level distribution (overall):
ALERT_MAX
YELLOW    19790
ORANGE     3937
RED          33
Name: count, dtype: int64


In [ ]:
def generate_message(disaster, level, district, weather):
    """Generate a human-readable alert message."""

    if disaster == "FIRE":
        if level == "RED":
            return (f"EMERGENCY: Extreme forest fire risk in {district}. "
                    f"Humidity critically low at {weather['humidity']:.0f}%, "
                    f"temperature {weather['temperature']:.1f}°C. "
                    f"All forest activities banned. Fire brigades on standby.")
        elif level == "ORANGE":
            return (f"WARNING: High forest fire risk in {district}. "
                    f"Low humidity ({weather['humidity']:.0f}%) and dry conditions. "
                    f"Avoid open burning. Increased patrolling advised.")
        else:
            return (f"WATCH: Moderate fire risk in {district}. "
                    f"Humidity at {weather['humidity']:.0f}%. Stay alert.")

    elif disaster == "FLOOD":
        if level == "RED":
            return (f"EMERGENCY: Extreme flood risk in {district}. "
                    f"Rainfall {weather['rainfall_mm']:.0f}mm today, "
                    f"3-day total {weather['rain_3day_cum']:.0f}mm. "
                    f"Evacuate low-lying areas immediately.")
        elif level == "ORANGE":
            return (f"WARNING: High flood risk in {district}. "
                    f"Heavy rainfall {weather['rainfall_mm']:.0f}mm/day. "
                    f"Avoid riverbanks and flood-prone areas.")
        else:
            return (f"WATCH: Elevated flood risk in {district}. "
                    f"Rainfall {weather['rainfall_mm']:.0f}mm. Monitor river levels.")

    elif disaster == "LANDSLIDE":
        if level == "RED":
            return (f"EMERGENCY: Extreme landslide risk in {district}. "
                    f"Soil saturation critical ({weather['soil_wetness']:.2f}), "
                    f"7-day rainfall avg {weather['rain_7day_avg']:.0f}mm/day. "
                    f"Close mountain roads. Evacuate slope areas.")
        elif level == "ORANGE":
            return (f"WARNING: High landslide risk in {district}. "
                    f"Sustained rainfall and saturated soil. "
                    f"Avoid hill roads and steep slopes.")
        else:
            return (f"WATCH: Landslide conditions developing in {district}. "
                    f"Soil wetness {weather['soil_wetness']:.2f}. Monitor slopes.")

    elif disaster == "HEATWAVE":
        if level == "RED":
            return (f"EMERGENCY: Extreme heat wave in {district}. "
                    f"Max temperature {weather['max_temp']:.1f}°C. "
                    f"Stay indoors, avoid outdoor work 11am–4pm. "
                    f"Hydration critical.")
        elif level == "ORANGE":
            return (f"WARNING: Severe heat wave in {district}. "
                    f"Temperature {weather['max_temp']:.1f}°C. "
                    f"Limit outdoor exposure. Check on elderly and children.")
        else:
            return (f"WATCH: Heat stress conditions in {district}. "
                    f"Temperature {weather['max_temp']:.1f}°C. Stay hydrated.")

    return f"{level} alert for {disaster} in {district}."

# ── Regenerate alerts with messages (now that function is defined) ──
alerts = []

for (date, district), grp in df.groupby(["DATE", "DISTRICT"]):
    for disaster in ["FIRE", "FLOOD", "LANDSLIDE", "HEATWAVE"]:
        score_col   = f"FINAL_{disaster}_SCORE"
        alert_col   = f"ALERT_{disaster}"
        worst_idx   = grp[score_col].idxmax()
        worst_score = grp.loc[worst_idx, score_col]
        alert_level = grp.loc[worst_idx, alert_col]
        ml_conf     = grp.loc[worst_idx, f"ML_{disaster}_CONF"] if f"ML_{disaster}_CONF" in grp.columns else 0.9

        if alert_level == "GREEN":
            continue

        level_info = ALERT_LEVELS[alert_level]
        weather = {
            "temperature"   : round(float(grp["T2M"].mean()), 1),
            "max_temp"      : round(float(grp["T2M_MAX"].mean()), 1),
            "humidity"      : round(float(grp["RH2M"].mean()), 1),
            "rainfall_mm"   : round(float(grp["PRECTOTCORR"].mean()), 1),
            "wind_ms"       : round(float(grp["WS10M"].mean()), 1),
            "soil_wetness"  : round(float(grp["GWETROOT"].mean()), 3),
            "rain_3day_cum" : round(float(grp["PRECIP_3D_CUM"].mean()), 1),
            "rain_7day_avg" : round(float(grp["PRECIP_7D"].mean()), 1),
        }
        msg = generate_message(disaster, alert_level, district, weather)

        alerts.append({
            "alert_id"      : f"{date.strftime('%Y%m%d')}_{district}_{disaster}_{alert_level}",
            "date"          : date.strftime("%Y-%m-%d"),
            "district"      : district,
            "disaster_type" : DISASTER_NAMES[disaster],
            "disaster_code" : disaster,
            "alert_level"   : alert_level,
            "alert_emoji"   : level_info["emoji"],
            "alert_label"   : level_info["label"],
            "risk_score"    : round(float(worst_score), 3),
            "ml_confidence" : round(float(ml_conf), 3),
            "action"        : level_info["action"],
            "message"       : msg,
            "weather"       : weather,
            "lat"           : float(grp.loc[worst_idx, "LAT"]),
            "lon"           : float(grp.loc[worst_idx, "LON"]),
        })

df_alerts = pd.DataFrame(alerts)
print(f"Total alerts generated : {len(df_alerts)}")
print(f"\nAlert level breakdown:")
print(df_alerts["alert_level"].value_counts())
print(f"\nDisaster type breakdown:")
print(df_alerts["disaster_type"].value_counts())
print(f"\nTop 5 most alerted districts:")
print(df_alerts["district"].value_counts().head())

Total alerts generated : 14398

Alert level breakdown:
alert_level
YELLOW    12441
ORANGE     1926
RED          31
Name: count, dtype: int64

Disaster type breakdown:
disaster_type
Forest Fire    3600
Heat Wave      3600
Flood          3599
Landslide      3599
Name: count, dtype: int64

Top 5 most alerted districts:
district
Chamoli      1440
Champawat    1440
Pauri        1440
Dehradun     1440
Haridwar     1440
Name: count, dtype: int64


In [ ]:
#GENERATION OF STRUCTURED ALERTS
# One alert record per (date, district, disaster) that is YELLOW or above

alerts = []

# Group by date and district
for (date, district), grp in df.groupby(["DATE", "DISTRICT"]):

    for disaster in ["FIRE", "FLOOD", "LANDSLIDE", "HEATWAVE"]:
        score_col = f"FINAL_{disaster}_SCORE"
        alert_col = f"ALERT_{disaster}"

        # Take the worst cell in this district on this day
        worst_idx   = grp[score_col].idxmax()
        worst_score = grp.loc[worst_idx, score_col]
        alert_level = grp.loc[worst_idx, alert_col]
        ml_conf     = grp.loc[worst_idx, f"ML_{disaster}_CONF"] if f"ML_{disaster}_CONF" in grp.columns else 0.9

        # Only create alerts for YELLOW and above
        if alert_level == "GREEN":
            continue

        level_info = ALERT_LEVELS[alert_level]

        # Build weather context for this alert
        weather = {
            "temperature"   : round(float(grp["T2M"].mean()), 1),
            "max_temp"      : round(float(grp["T2M_MAX"].mean()), 1),
            "humidity"      : round(float(grp["RH2M"].mean()), 1),
            "rainfall_mm"   : round(float(grp["PRECTOTCORR"].mean()), 1),
            "wind_ms"       : round(float(grp["WS10M"].mean()), 1),
            "soil_wetness"  : round(float(grp["GWETROOT"].mean()), 3),
            "rain_3day_cum" : round(float(grp["PRECIP_3D_CUM"].mean()), 1),
            "rain_7day_avg" : round(float(grp["PRECIP_7D"].mean()), 1),
        }

        # Human-readable warning message
        msg = generate_message(disaster, alert_level, district, weather)

        alerts.append({
            "alert_id"      : f"{date.strftime('%Y%m%d')}_{district}_{disaster}_{alert_level}",
            "date"          : date.strftime("%Y-%m-%d"),
            "district"      : district,
            "disaster_type" : DISASTER_NAMES[disaster],
            "disaster_code" : disaster,
            "alert_level"   : alert_level,
            "alert_emoji"   : level_info["emoji"],
            "alert_label"   : level_info["label"],
            "risk_score"    : round(float(worst_score), 3),
            "ml_confidence" : round(float(ml_conf), 3),
            "action"        : level_info["action"],
            "message"       : msg,
            "weather"       : weather,
            "lat"           : float(grp.loc[worst_idx, "LAT"]),
            "lon"           : float(grp.loc[worst_idx, "LON"]),
        })

print(f"Total alerts generated : {len(alerts)}")

Total alerts generated : 14398


In [ ]:
#DISTRICT DAILY SUMMARY
# One row per district per day — worst risk across all disasters

district_daily = df.groupby(["DATE", "DISTRICT"]).agg(
    FIRE_SCORE      = ("FINAL_FIRE_SCORE",      "max"),
    FLOOD_SCORE     = ("FINAL_FLOOD_SCORE",     "max"),
    LANDSLIDE_SCORE = ("FINAL_LANDSLIDE_SCORE", "max"),
    HEATWAVE_SCORE  = ("FINAL_HEATWAVE_SCORE",  "max"),
    MAX_SCORE       = ("FINAL_MAX_SCORE",        "max"),
    TEMP            = ("T2M",                   "mean"),
    RAINFALL        = ("PRECTOTCORR",           "mean"),
    HUMIDITY        = ("RH2M",                  "mean"),
).reset_index()

district_daily["ALERT_LEVEL"] = district_daily["MAX_SCORE"].apply(get_alert_level)
district_daily["DATE"] = district_daily["DATE"].astype(str)

print(f"\nDistrict daily summary shape: {district_daily.shape}")
print(district_daily.head(5).to_string())


District daily summary shape: (3600, 11)
         DATE   DISTRICT  FIRE_SCORE  FLOOD_SCORE  LANDSLIDE_SCORE  HEATWAVE_SCORE  MAX_SCORE       TEMP  RAINFALL   HUMIDITY ALERT_LEVEL
0  2025-03-01  Bageshwar         1.6          1.8              1.8             1.8        1.8   7.750000  0.000000  38.380000      YELLOW
1  2025-03-01    Chamoli         1.8          1.8              1.8             1.8        1.8  -2.926429  0.002143  49.936786      YELLOW
2  2025-03-01  Champawat         1.8          1.8              1.8             1.8        1.8  19.230000  0.001250  40.165625      YELLOW
3  2025-03-01   Dehradun         1.6          1.8              1.8             1.8        1.8  15.335000  0.000000  39.042500      YELLOW
4  2025-03-01   Haridwar         1.6          1.8              1.8             3.2        3.2  20.777143  0.000000  36.820000      ORANGE


In [ ]:
#MOST RECENT DAY SNAPSHOT
# What is the current alert status for each district?

latest_date  = df["DATE"].max()
latest_data  = df[df["DATE"] == latest_date]

print(f"\n{'='*60}")
print(f"  CURRENT ALERT STATUS — {latest_date.strftime('%d %b %Y')}")
print(f"{'='*60}")

latest_district = latest_data.groupby("DISTRICT").agg(
    FIRE_SCORE      = ("FINAL_FIRE_SCORE",      "max"),
    FLOOD_SCORE     = ("FINAL_FLOOD_SCORE",     "max"),
    LANDSLIDE_SCORE = ("FINAL_LANDSLIDE_SCORE", "max"),
    HEATWAVE_SCORE  = ("FINAL_HEATWAVE_SCORE",  "max"),
    MAX_SCORE       = ("FINAL_MAX_SCORE",        "max"),
).reset_index()

latest_district["ALERT"] = latest_district["MAX_SCORE"].apply(get_alert_level)

for _, row in latest_district.sort_values("MAX_SCORE", ascending=False).iterrows():
    lvl  = row["ALERT"]
    info = ALERT_LEVELS[lvl]
    print(f"  {info['emoji']} {row['DISTRICT']:<15} "
          f"Fire:{row['FIRE_SCORE']:.1f}  "
          f"Flood:{row['FLOOD_SCORE']:.1f}  "
          f"Slide:{row['LANDSLIDE_SCORE']:.1f}  "
          f"Heat:{row['HEATWAVE_SCORE']:.1f}  "
          f"→ {info['label']}")

#Saving the alerts

# ── Full alerts JSON (for dashboard) ──
alerts_json_path = "alerts/all_alerts.json"
with open(alerts_json_path, "w") as f:
    json.dump(alerts, f, indent=2)

# ── District daily summary CSV ──
district_csv_path = "alerts/district_daily_summary.csv"
district_daily.to_csv(district_csv_path, index=False)

# ── Latest snapshot JSON ──
latest_snap = latest_district.to_dict(orient="records")
for rec in latest_snap:
    rec["alert_info"] = ALERT_LEVELS[rec["ALERT"]]
with open("alerts/latest_snapshot.json", "w") as f:
    json.dump({
        "date"      : latest_date.strftime("%Y-%m-%d"),
        "districts" : latest_snap
    }, f, indent=2)

# ── RED alerts only ──
red_alerts = [a for a in alerts if a["alert_level"] == "RED"]
with open("alerts/red_alerts.json", "w") as f:
    json.dump(red_alerts, f, indent=2)

print(f"\nFiles saved in ./alerts/:")
for fname in os.listdir("alerts"):
    size = os.path.getsize(f"alerts/{fname}") / 1024
    print(f"  {fname:<40} ({size:.0f} KB)")


# Count by level
level_counts = df_alerts["alert_level"].value_counts().to_dict()
red_count    = level_counts.get("RED", 0)
orange_count = level_counts.get("ORANGE", 0)
yellow_count = level_counts.get("YELLOW", 0)

# Top RED alerts
top_red = [a for a in alerts if a["alert_level"] == "RED"][:10]

alert_rows = ""
for a in top_red:
    alert_rows += f"""
    <tr>
      <td>{a['date']}</td>
      <td><b>{a['district']}</b></td>
      <td>{a['alert_emoji']} {a['disaster_type']}</td>
      <td style="color:#c1121f;font-weight:700">{a['alert_label']}</td>
      <td>{a['risk_score']:.2f}</td>
      <td style="font-size:12px">{a['message'][:80]}...</td>
    </tr>"""

# Latest status cards
status_cards = ""
for row in latest_district.sort_values("MAX_SCORE", ascending=False).to_dict("records"):
    lvl   = row["ALERT"]
    info  = ALERT_LEVELS[lvl]
    color = {"GREEN":"#2d6a4f","YELLOW":"#e9c46a","ORANGE":"#f4a261","RED":"#c1121f"}[lvl]
    status_cards += f"""
    <div class="dcard" style="border-left:4px solid {color}">
      <div class="dname">{row['DISTRICT']}</div>
      <div class="dalert" style="color:{color}">{info['emoji']} {info['label']}</div>
      <div class="dscores">
        🔥{row['FIRE_SCORE']:.1f} &nbsp;
        🌊{row['FLOOD_SCORE']:.1f} &nbsp;
        ⛰️{row['LANDSLIDE_SCORE']:.1f} &nbsp;
        🌡️{row['HEATWAVE_SCORE']:.1f}
      </div>
    </div>"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<title>Uttarakhand Disaster Alert Bulletin</title>
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;800&family=IBM+Plex+Mono:wght@400;600&display=swap" rel="stylesheet"/>
<style>
:root{{--bg:#0b0f1a;--surf:#131929;--bdr:#1e2d45;--acc:#00d9a3;--acc2:#4da6ff;--txt:#e2eaf6;--mut:#7a91b0;}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:'Syne',sans-serif;background:var(--bg);color:var(--txt);}}
.hero{{background:linear-gradient(135deg,#0b1628,#1a0a0a 60%,#0d0d1a);
  border-bottom:1px solid var(--bdr);padding:40px 48px;}}
.htag{{font-family:'IBM Plex Mono',monospace;font-size:11px;letter-spacing:3px;
  color:#ff4444;text-transform:uppercase;margin-bottom:10px;}}
.hero h1{{font-size:clamp(20px,3.5vw,36px);font-weight:800;margin-bottom:6px;}}
.hero h1 em{{color:#ff4444;font-style:normal;}}
.hsub{{color:var(--mut);font-size:13px;font-family:'IBM Plex Mono',monospace;}}
.sec{{padding:32px 48px;border-bottom:1px solid var(--bdr);}}
.stitle{{font-size:11px;letter-spacing:3px;text-transform:uppercase;
  color:var(--acc);font-family:'IBM Plex Mono',monospace;margin-bottom:20px;}}
.summary-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr));gap:16px;}}
.scard{{background:var(--surf);border:1px solid var(--bdr);border-radius:8px;
  padding:20px;text-align:center;}}
.snum{{font-size:36px;font-weight:800;}}
.slbl{{font-size:11px;color:var(--mut);margin-top:4px;letter-spacing:1px;}}
.red{{color:#c1121f;}} .orange{{color:#f4a261;}} .yellow{{color:#e9c46a;}} .green{{color:#2d6a4f;}}
.district-grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:12px;}}
.dcard{{background:var(--surf);border-radius:8px;padding:16px;border-left:4px solid #333;}}
.dname{{font-size:14px;font-weight:700;margin-bottom:4px;}}
.dalert{{font-size:13px;font-weight:600;margin-bottom:6px;}}
.dscores{{font-size:11px;color:var(--mut);font-family:'IBM Plex Mono',monospace;}}
.twrap{{overflow-x:auto;}}
table{{width:100%;border-collapse:collapse;font-size:12px;}}
thead tr{{background:#1a0505;border-bottom:2px solid #c1121f;}}
th{{padding:10px 14px;text-align:left;font-size:10px;letter-spacing:1px;
  text-transform:uppercase;color:#ff6666;font-weight:600;}}
td{{padding:10px 14px;border-bottom:1px solid var(--bdr);vertical-align:top;}}
tbody tr:hover{{background:#12233a;}}
.foot{{padding:20px 48px;color:var(--mut);font-size:11px;font-family:'IBM Plex Mono',monospace;}}
</style>
</head>
<body>
<div class="hero">
  <div class="htag">⚠ DISASTER ALERT BULLETIN — UTTARAKHAND</div>
  <h1>Disaster <em>Risk Alert</em> Report</h1>
  <div class="hsub">NASA POWER MERRA-2 + ML Prediction Engine &nbsp;|&nbsp; Mar 2025 – Mar 2026</div>
</div>

<div class="sec">
  <div class="stitle">Alert Summary</div>
  <div class="summary-grid">
    <div class="scard"><div class="snum red">{red_count}</div><div class="slbl">🔴 Red (Emergency)</div></div>
    <div class="scard"><div class="snum orange">{orange_count}</div><div class="slbl">🟠 Orange (Warning)</div></div>
    <div class="scard"><div class="snum yellow">{yellow_count}</div><div class="slbl">🟡 Yellow (Watch)</div></div>
    <div class="scard"><div class="snum" style="color:var(--acc2)">{len(alerts)}</div><div class="slbl">Total Alerts</div></div>
    <div class="scard"><div class="snum" style="color:var(--acc)">{len(df_alerts['district'].unique())}</div><div class="slbl">Districts Affected</div></div>
    <div class="scard"><div class="snum" style="color:var(--mut)">{len(df_alerts['date'].unique())}</div><div class="slbl">Alert Days</div></div>
  </div>
</div>

<div class="sec">
  <div class="stitle">Current District Status — {latest_date.strftime('%d %b %Y')}</div>
  <div class="district-grid">{status_cards}</div>
</div>

<div class="sec">
  <div class="stitle">Top Red (Emergency) Alerts</div>
  <div class="twrap">
    <table>
      <thead>
        <tr><th>Date</th><th>District</th><th>Disaster</th><th>Level</th><th>Score</th><th>Message</th></tr>
      </thead>
      <tbody>{alert_rows}</tbody>
    </table>
  </div>
</div>

<div class="foot">
  Uttarakhand Disaster Prediction System &middot; Step 5: Alert Engine &middot; NASA POWER MERRA-2
</div>
</body>
</html>"""

html_path = "alerts/alert_bulletin.html"
with open(html_path, "w") as f:
    f.write(html)

print(f"\n{'='*60}")
print(f"  STEP 5 COMPLETE ✅")
print(f"  Total alerts   : {len(alerts)}")
print(f"  🔴 RED         : {red_count}")
print(f"  🟠 ORANGE      : {orange_count}")
print(f"  🟡 YELLOW      : {yellow_count}")
print(f"  HTML bulletin  : alerts/alert_bulletin.html")
print(f"{'='*60}")
print(f"\nDownload & open alert_bulletin.html in browser!")


  CURRENT ALERT STATUS — 28 Feb 2026
  🟠 Uttarkashi      Fire:1.8  Flood:1.8  Slide:3.2  Heat:1.8  → Warning
  🟡 Bageshwar       Fire:1.6  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Chamoli         Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Champawat       Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Haridwar        Fire:1.6  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Dehradun        Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Nainital        Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Pauri           Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Pithoragarh     Fire:1.8  Flood:1.8  Slide:1.8  Heat:1.8  → Watch
  🟡 Tehri           Fire:1.8  Flood:1.8  Slide:1.6  Heat:1.8  → Watch

Files saved in ./alerts/:
  all_alerts.json                          (10217 KB)
  district_daily_summary.csv               (284 KB)
  latest_snapshot.json                     (4 KB)
  red_alerts.json                          (24 KB)

  STEP 5 COMPLETE ✅
  Tot

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# ── Load ML predictions ──
df = pd.read_csv("ml_results/uttarakhand_ml_predictions.csv")
df["YEAR"] = df["MO"].apply(lambda m: 2025 if m >= 3 else 2026)
df["DATE"] = pd.to_datetime(
    df["YEAR"].astype(str) + "-" + df["DAY_OF_YEAR"].astype(str),
    format="%Y-%j"
)

# ── Load alerts ──
with open("alerts/all_alerts.json") as f:
    all_alerts = json.load(f)

with open("alerts/latest_snapshot.json") as f:
    snapshot = json.load(f)

print(f"ML data  : {len(df):,} rows")
print(f"Alerts   : {len(all_alerts):,}")
print(f"Snapshot : {snapshot['date']}")

ML data  : 23,760 rows
Alerts   : 14,398
Snapshot : 2026-02-28


In [ ]:
DISTRICTS = {
    "Dehradun"    : (30.316, 78.032),
    "Haridwar"    : (29.946, 78.162),
    "Pauri"       : (29.771, 78.780),
    "Tehri"       : (30.378, 78.480),
    "Uttarkashi"  : (30.726, 78.440),
    "Chamoli"     : (30.401, 79.321),
    "Rudraprayag" : (30.284, 78.981),
    "Nainital"    : (29.381, 79.463),
    "Almora"      : (29.597, 79.646),
    "Bageshwar"   : (29.838, 79.770),
    "Pithoragarh" : (29.582, 80.218),
    "Champawat"   : (29.335, 80.091),
}

def nearest_district(lat, lon):
    return min(DISTRICTS, key=lambda d: (lat-DISTRICTS[d][0])**2 + (lon-DISTRICTS[d][1])**2)

df["DISTRICT"] = df.apply(lambda r: nearest_district(r["LAT"], r["LON"]), axis=1)

# ── Monthly aggregates ──
monthly = df.groupby(["YEAR","MO"]).agg(
    FIRE      = ("FINAL_FIRE_SCORE",      "mean"),
    FLOOD     = ("FINAL_FLOOD_SCORE",     "mean"),
    LANDSLIDE = ("FINAL_LANDSLIDE_SCORE", "mean"),
    HEATWAVE  = ("FINAL_HEATWAVE_SCORE",  "mean"),
    TEMP      = ("T2M",                   "mean"),
    RAIN      = ("PRECTOTCORR",           "mean"),
    HUMIDITY  = ("RH2M",                  "mean"),
).reset_index().round(3)

import calendar
monthly["LABEL"] = monthly.apply(
    lambda r: f"{calendar.month_abbr[int(r['MO'])]} {int(r['YEAR'])}", axis=1
)
monthly = monthly.sort_values(["YEAR","MO"])

# ── Grid averages per cell ──
grid_avg = df.groupby(["LAT","LON"]).agg(
    FIRE      = ("FINAL_FIRE_SCORE",      "mean"),
    FLOOD     = ("FINAL_FLOOD_SCORE",     "mean"),
    LANDSLIDE = ("FINAL_LANDSLIDE_SCORE", "mean"),
    HEATWAVE  = ("FINAL_HEATWAVE_SCORE",  "mean"),
    MAX       = ("FINAL_MAX_SCORE",        "mean"),
).reset_index().round(3)

# ── District summary ──
district_summary = df.groupby("DISTRICT").agg(
    FIRE      = ("FINAL_FIRE_SCORE",      "mean"),
    FLOOD     = ("FINAL_FLOOD_SCORE",     "mean"),
    LANDSLIDE = ("FINAL_LANDSLIDE_SCORE", "mean"),
    HEATWAVE  = ("FINAL_HEATWAVE_SCORE",  "mean"),
    MAX       = ("FINAL_MAX_SCORE",        "mean"),
    TEMP      = ("T2M",                   "mean"),
    RAIN      = ("PRECTOTCORR",           "mean"),
).reset_index().round(3)

# ── Alert counts ──
alert_counts = {"RED":0,"ORANGE":0,"YELLOW":0,"GREEN":0}
for a in all_alerts:
    alert_counts[a["alert_level"]] = alert_counts.get(a["alert_level"],0) + 1

# ── Top RED alerts ──
red_alerts = [a for a in all_alerts if a["alert_level"]=="RED"][:8]

# ── Latest snapshot districts ──
latest_districts = sorted(
    snapshot["districts"],
    key=lambda x: x["MAX_SCORE"], reverse=True
)

def score_color(s):
    if s >= 3.5: return "#ef233c"
    elif s >= 2.5: return "#f77f00"
    elif s >= 1.5: return "#f4d03f"
    else: return "#2dc653"

def score_label(s):
    if s >= 3.5: return "EMERGENCY"
    elif s >= 2.5: return "WARNING"
    elif s >= 1.5: return "WATCH"
    else: return "SAFE"

def score_emoji(s):
    if s >= 3.5: return "🔴"
    elif s >= 2.5: return "🟠"
    elif s >= 1.5: return "🟡"
    else: return "🟢"

# ── Serialize to JSON strings for JS ──
monthly_json  = monthly.to_json(orient="records")
grid_json     = grid_avg.to_json(orient="records")
district_json = district_summary.to_json(orient="records")

print("Data prepared ✅")
print(f"Monthly points : {len(monthly)}")
print(f"Grid cells     : {len(grid_avg)}")
print(f"Districts      : {len(district_summary)}")


Data prepared ✅
Monthly points : 12
Grid cells     : 66
Districts      : 10


In [ ]:
# CELL 3 — BUILD DISTRICT STATUS HTML (FIXED)

district_cards_html = ""
for d in latest_districts:
    # handle both key name formats
    s     = d.get("MAX_SCORE") or d.get("MAX", 0) or 0
    fire  = d.get("FIRE_SCORE") or d.get("FIRE", 0) or 0
    flood = d.get("FLOOD_SCORE") or d.get("FLOOD", 0) or 0
    slide = d.get("LANDSLIDE_SCORE") or d.get("LANDSLIDE", 0) or 0
    heat  = d.get("HEATWAVE_SCORE") or d.get("HEATWAVE", 0) or 0
    name  = d.get("DISTRICT","Unknown")

    color = score_color(s)
    label = score_label(s)
    emoji = score_emoji(s)

    district_cards_html += f"""
        <div class="dcard" onclick="selectDistrict('{name}')" id="dc-{name.replace(' ','-')}">
          <div class="dcard-top">
            <span class="dname">{name}</span>
            <span class="dbadge" style="background:{color}22;color:{color};border:1px solid {color}44">{emoji} {label}</span>
          </div>
          <div class="dbar-wrap">
            <div class="dbar" style="width:{min(s/4*100,100):.0f}%;background:{color}"></div>
          </div>
          <div class="dscores">
            <span>🔥 {fire:.1f}</span>
            <span>🌊 {flood:.1f}</span>
            <span>⛰️ {slide:.1f}</span>
            <span>🌡️ {heat:.1f}</span>
          </div>
        </div>"""

# Alert rows
alert_rows_html = ""
for a in red_alerts:
    alert_rows_html += f"""
        <div class="arow">
          <div class="adate">{a['date']}</div>
          <div class="adistrict">{a['district']}</div>
          <div class="atype">{a['disaster_type']}</div>
          <div class="abadge" style="background:#ef233c22;color:#ef233c;border:1px solid #ef233c44">
            🔴 {a['alert_label']}
          </div>
          <div class="amsg">{a['message'][:90]}...</div>
        </div>"""

print("HTML components built ✅")
print(f"District cards : {district_cards_html.count('dcard')}")
print(f"Alert rows     : {alert_rows_html.count('arow')}")

# CELL 4 — BUILD FULL DASHBOARD HTML

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>Uttarakhand Disaster Warning System</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet"/>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
/* ── RESET & BASE ── */
:root{{
  --bg       : #060910;
  --surf1    : #0d1117;
  --surf2    : #161b22;
  --surf3    : #1c2333;
  --bdr      : #21262d;
  --acc      : #00ff9d;
  --acc-dim  : #00ff9d22;
  --acc2     : #3d9bff;
  --acc2-dim : #3d9bff22;
  --red      : #ef233c;
  --orange   : #f77f00;
  --yellow   : #f4d03f;
  --green    : #2dc653;
  --txt      : #e6edf3;
  --txt2     : #8b949e;
  --txt3     : #484f58;
  --mono     : 'JetBrains Mono', monospace;
  --sans     : 'Space Grotesk', sans-serif;
}}
*{{box-sizing:border-box;margin:0;padding:0;}}
html{{scroll-behavior:smooth;}}
body{{
  font-family:var(--sans);
  background:var(--bg);
  color:var(--txt);
  min-height:100vh;
  overflow-x:hidden;
}}

/* ── SCROLLBAR ── */
::-webkit-scrollbar{{width:6px;height:6px;}}
::-webkit-scrollbar-track{{background:var(--surf1);}}
::-webkit-scrollbar-thumb{{background:var(--bdr);border-radius:3px;}}

/* ── TOP NAV ── */
.topnav{{
  position:fixed;top:0;left:0;right:0;z-index:1000;
  background:rgba(6,9,16,0.85);
  backdrop-filter:blur(20px);
  border-bottom:1px solid var(--bdr);
  display:flex;align-items:center;justify-content:space-between;
  padding:0 28px;height:56px;
}}
.nav-brand{{
  display:flex;align-items:center;gap:10px;
  font-size:15px;font-weight:700;letter-spacing:-0.3px;
}}
.nav-dot{{
  width:8px;height:8px;border-radius:50%;
  background:var(--acc);
  box-shadow:0 0 12px var(--acc);
  animation:pulse 2s infinite;
}}
@keyframes pulse{{0%,100%{{opacity:1;transform:scale(1);}}50%{{opacity:0.6;transform:scale(1.3);}}}}
.nav-right{{
  display:flex;align-items:center;gap:16px;
  font-size:12px;font-family:var(--mono);color:var(--txt2);
}}
.live-tag{{
  background:var(--red);color:white;
  font-size:10px;font-weight:700;letter-spacing:2px;
  padding:3px 8px;border-radius:3px;
  animation:blink 1.5s infinite;
}}
@keyframes blink{{0%,100%{{opacity:1;}}50%{{opacity:0.5;}}}}

/* ── HERO ── */
.hero{{
  padding:100px 40px 48px;
  background:
    radial-gradient(ellipse at 20% 50%, rgba(0,255,157,0.04) 0%, transparent 60%),
    radial-gradient(ellipse at 80% 20%, rgba(61,155,255,0.05) 0%, transparent 60%),
    var(--bg);
  border-bottom:1px solid var(--bdr);
  position:relative;overflow:hidden;
}}
.hero::after{{
  content:'';position:absolute;
  bottom:-1px;left:0;right:0;height:1px;
  background:linear-gradient(90deg,transparent,var(--acc),transparent);
}}
.hero-eyebrow{{
  font-family:var(--mono);font-size:11px;letter-spacing:4px;
  color:var(--acc);text-transform:uppercase;margin-bottom:16px;
  display:flex;align-items:center;gap:10px;
}}
.hero-eyebrow::before{{
  content:'';width:24px;height:1px;background:var(--acc);
}}
.hero h1{{
  font-size:clamp(28px,5vw,56px);font-weight:700;
  line-height:1.05;letter-spacing:-1.5px;
  margin-bottom:16px;
}}
.hero h1 .hl{{
  background:linear-gradient(135deg,var(--acc),var(--acc2));
  -webkit-background-clip:text;-webkit-text-fill-color:transparent;
}}
.hero-sub{{
  font-size:15px;color:var(--txt2);max-width:600px;line-height:1.6;
  margin-bottom:32px;
}}
.hero-stats{{
  display:flex;flex-wrap:wrap;gap:32px;
}}
.hstat{{border-left:2px solid var(--bdr);padding-left:16px;}}
.hstat-num{{font-size:28px;font-weight:700;font-family:var(--mono);color:var(--acc);}}
.hstat-lbl{{font-size:11px;color:var(--txt2);letter-spacing:1px;text-transform:uppercase;margin-top:2px;}}

/* ── ALERT BANNER ── */
.alert-banner{{
  background:linear-gradient(90deg,rgba(239,35,60,0.12),rgba(239,35,60,0.04));
  border-bottom:1px solid rgba(239,35,60,0.2);
  padding:10px 40px;
  display:flex;align-items:center;gap:12px;
  font-size:13px;font-family:var(--mono);
  overflow:hidden;
}}
.ab-label{{
  background:var(--red);color:white;
  font-size:10px;font-weight:700;letter-spacing:2px;
  padding:3px 10px;border-radius:3px;white-space:nowrap;
}}
.ab-scroll{{
  white-space:nowrap;
  animation:scrolltxt 30s linear infinite;
  color:var(--txt2);
}}
@keyframes scrolltxt{{from{{transform:translateX(0);}}to{{transform:translateX(-50%);}}}}

/* ── LAYOUT ── */
.main{{padding:32px 40px;max-width:1600px;margin:0 auto;}}
.section-title{{
  font-size:11px;font-family:var(--mono);letter-spacing:3px;
  text-transform:uppercase;color:var(--acc);
  margin-bottom:20px;display:flex;align-items:center;gap:10px;
}}
.section-title::after{{content:'';flex:1;height:1px;background:var(--bdr);}}
.row{{display:grid;gap:20px;margin-bottom:32px;}}
.row-2{{grid-template-columns:1fr 1fr;}}
.row-3{{grid-template-columns:1fr 1fr 1fr;}}
.row-13{{grid-template-columns:1fr 3fr;}}
.row-31{{grid-template-columns:3fr 1fr;}}
@media(max-width:900px){{
  .row-2,.row-3,.row-13,.row-31{{grid-template-columns:1fr;}}
  .main{{padding:20px;}}
}}

/* ── CARDS ── */
.card{{
  background:var(--surf1);
  border:1px solid var(--bdr);
  border-radius:12px;
  padding:24px;
  position:relative;overflow:hidden;
}}
.card-title{{
  font-size:12px;font-family:var(--mono);letter-spacing:2px;
  text-transform:uppercase;color:var(--txt2);
  margin-bottom:16px;
}}
.card-accent{{border-top:2px solid var(--acc);}}
.card-red{{border-top:2px solid var(--red);}}
.card-blue{{border-top:2px solid var(--acc2);}}

/* ── STAT TILES ── */
.stat-grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:32px;}}
@media(max-width:700px){{.stat-grid{{grid-template-columns:repeat(2,1fr);}}}}
.stat-tile{{
  background:var(--surf1);border:1px solid var(--bdr);border-radius:10px;
  padding:20px;position:relative;overflow:hidden;
  transition:border-color 0.2s;cursor:default;
}}
.stat-tile:hover{{border-color:var(--acc);}}
.st-val{{font-size:32px;font-weight:700;font-family:var(--mono);line-height:1;}}
.st-lbl{{font-size:11px;color:var(--txt2);margin-top:6px;letter-spacing:1px;text-transform:uppercase;}}
.st-sub{{font-size:11px;color:var(--txt3);margin-top:4px;font-family:var(--mono);}}

/* ── DISTRICT CARDS ── */
.district-list{{display:flex;flex-direction:column;gap:8px;max-height:480px;overflow-y:auto;}}
.dcard{{
  background:var(--surf2);border:1px solid var(--bdr);border-radius:8px;
  padding:14px 16px;cursor:pointer;transition:all 0.15s;
}}
.dcard:hover,.dcard.active{{background:var(--surf3);border-color:var(--acc2);}}
.dcard-top{{display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;}}
.dname{{font-size:13px;font-weight:600;}}
.dbadge{{font-size:10px;font-family:var(--mono);padding:3px 8px;border-radius:20px;letter-spacing:1px;}}
.dbar-wrap{{height:3px;background:var(--bdr);border-radius:2px;margin-bottom:8px;}}
.dbar{{height:3px;border-radius:2px;transition:width 0.5s;}}
.dscores{{display:flex;gap:12px;font-size:11px;font-family:var(--mono);color:var(--txt2);}}

/* ── CHART CONTAINER ── */
.chart-wrap{{position:relative;height:260px;}}
.chart-wrap-tall{{position:relative;height:340px;}}

/* ── ALERT ROWS ── */
.alerts-list{{display:flex;flex-direction:column;gap:8px;max-height:400px;overflow-y:auto;}}
.arow{{
  background:var(--surf2);border:1px solid var(--bdr);border-radius:8px;
  padding:14px 16px;display:grid;
  grid-template-columns:90px 110px 110px 110px 1fr;
  gap:12px;align-items:center;font-size:12px;
  transition:background 0.15s;
}}
.arow:hover{{background:var(--surf3);}}
.adate{{font-family:var(--mono);color:var(--txt2);}}
.adistrict{{font-weight:600;}}
.atype{{color:var(--txt2);}}
.abadge{{font-size:10px;font-family:var(--mono);padding:3px 8px;border-radius:20px;letter-spacing:1px;white-space:nowrap;}}
.amsg{{color:var(--txt2);font-size:11px;}}
@media(max-width:900px){{
  .arow{{grid-template-columns:1fr;}}
  .adate,.atype{{display:none;}}
}}

/* ── HEATMAP GRID ── */
.heatmap-grid{{display:grid;gap:4px;}}
.hm-cell{{
  border-radius:4px;cursor:pointer;
  display:flex;align-items:center;justify-content:center;
  font-size:10px;font-family:var(--mono);color:rgba(255,255,255,0.6);
  transition:transform 0.15s,filter 0.15s;
  aspect-ratio:1;
}}
.hm-cell:hover{{transform:scale(1.1);filter:brightness(1.3);z-index:10;position:relative;}}

/* ── TABS ── */
.tabs{{display:flex;gap:4px;margin-bottom:20px;border-bottom:1px solid var(--bdr);padding-bottom:0;}}
.tab{{
  padding:8px 16px;font-size:12px;font-family:var(--mono);
  color:var(--txt2);cursor:pointer;border-bottom:2px solid transparent;
  transition:all 0.15s;letter-spacing:1px;text-transform:uppercase;
  margin-bottom:-1px;
}}
.tab:hover{{color:var(--txt);}}
.tab.active{{color:var(--acc);border-bottom-color:var(--acc);}}

/* ── RISK METER ── */
.risk-meter{{
  display:flex;height:8px;border-radius:4px;overflow:hidden;gap:2px;
  margin:12px 0;
}}
.rm-seg{{flex:1;border-radius:2px;}}

/* ── FOOTER ── */
.footer{{
  border-top:1px solid var(--bdr);
  padding:24px 40px;
  display:flex;justify-content:space-between;align-items:center;
  font-size:12px;font-family:var(--mono);color:var(--txt3);
  flex-wrap:wrap;gap:12px;
}}
</style>
</head>
<body>

<!-- TOP NAV -->
<nav class="topnav">
  <div class="nav-brand">
    <div class="nav-dot"></div>
    🏔️ Uttarakhand Disaster Warning System
  </div>
  <div class="nav-right">
    <span class="live-tag">LIVE</span>
    <span>NASA POWER MERRA-2</span>
    <span id="clock"></span>
  </div>
</nav>

<!-- ALERT BANNER -->
<div class="alert-banner">
  <div class="ab-label">⚠ ALERTS</div>
  <div class="ab-scroll" id="alert-scroll">
    {'&nbsp;&nbsp;&nbsp;|&nbsp;&nbsp;&nbsp;'.join([f"🔴 {a['district']} — {a['disaster_type']} EMERGENCY ({a['date']})" for a in red_alerts[:6]])}
    &nbsp;&nbsp;&nbsp;|&nbsp;&nbsp;&nbsp;
    {'&nbsp;&nbsp;&nbsp;|&nbsp;&nbsp;&nbsp;'.join([f"🔴 {a['district']} — {a['disaster_type']} EMERGENCY ({a['date']})" for a in red_alerts[:6]])}
  </div>
</div>

<!-- HERO -->
<div class="hero">
  <div class="hero-eyebrow">Disaster Prediction Engine · Uttarakhand</div>
  <h1>Real-Time <span class="hl">Disaster Risk</span><br/>Warning System</h1>
  <p class="hero-sub">
    AI-powered early warning system for Uttarakhand combining NASA POWER satellite
    data with Random Forest ML models to predict Forest Fire, Flood, Landslide
    and Heat Wave risks across 9×9 grid cells.
  </p>
  <div class="hero-stats">
    <div class="hstat">
      <div class="hstat-num">24,156</div>
      <div class="hstat-lbl">Data Records</div>
    </div>
    <div class="hstat">
      <div class="hstat-num">99%+</div>
      <div class="hstat-lbl">ML Accuracy</div>
    </div>
    <div class="hstat">
      <div class="hstat-num">{alert_counts.get('RED',0)}</div>
      <div class="hstat-lbl">Red Alerts</div>
    </div>
    <div class="hstat">
      <div class="hstat-num">8</div>
      <div class="hstat-lbl">Parameters</div>
    </div>
    <div class="hstat">
      <div class="hstat-num">Mar 25–26</div>
      <div class="hstat-lbl">Date Range</div>
    </div>
  </div>
</div>

<!-- MAIN CONTENT -->
<div class="main">

  <!-- STAT TILES -->
  <div class="stat-grid">
    <div class="stat-tile">
      <div class="st-val" style="color:var(--red)">{alert_counts.get('RED',0)}</div>
      <div class="st-lbl">🔴 Red Alerts</div>
      <div class="st-sub">Emergency level</div>
    </div>
    <div class="stat-tile">
      <div class="st-val" style="color:var(--orange)">{alert_counts.get('ORANGE',0)}</div>
      <div class="st-lbl">🟠 Orange Alerts</div>
      <div class="st-sub">Warning level</div>
    </div>
    <div class="stat-tile">
      <div class="st-val" style="color:var(--yellow)">{alert_counts.get('YELLOW',0)}</div>
      <div class="st-lbl">🟡 Yellow Alerts</div>
      <div class="st-sub">Watch level</div>
    </div>
    <div class="stat-tile">
      <div class="st-val" style="color:var(--acc)">{len(all_alerts)}</div>
      <div class="st-lbl">Total Alerts</div>
      <div class="st-sub">All districts</div>
    </div>
  </div>

  <!-- DISTRICT STATUS + MONTHLY CHART -->
  <div class="section-title">District Status</div>
  <div class="row row-13" style="margin-bottom:32px">

    <!-- District List -->
    <div class="card card-blue">
      <div class="card-title">Current Risk — All Districts</div>
      <div class="district-list">
        {district_cards_html}
      </div>
    </div>

    <!-- Monthly Risk Chart -->
    <div class="card card-accent">
      <div class="card-title">Monthly Risk Trends — All Disasters</div>
      <div class="tabs">
        <div class="tab active" onclick="switchTab('risk')">Risk Scores</div>
        <div class="tab" onclick="switchTab('weather')">Weather</div>
      </div>
      <div class="chart-wrap-tall" id="chart-risk">
        <canvas id="monthlyRiskChart"></canvas>
      </div>
      <div class="chart-wrap-tall" id="chart-weather" style="display:none">
        <canvas id="weatherChart"></canvas>
      </div>
    </div>
  </div>

  <!-- RISK HEATMAP + DISASTER BREAKDOWN -->
  <div class="section-title">Spatial Risk Analysis</div>
  <div class="row row-2" style="margin-bottom:32px">

    <!-- Heatmap -->
    <div class="card">
      <div class="card-title">Grid Risk Heatmap — Select Layer</div>
      <div class="tabs">
        <div class="tab active" onclick="switchHeatmap('MAX')">Overall</div>
        <div class="tab" onclick="switchHeatmap('FIRE')">🔥 Fire</div>
        <div class="tab" onclick="switchHeatmap('FLOOD')">🌊 Flood</div>
        <div class="tab" onclick="switchHeatmap('LANDSLIDE')">⛰️ Slide</div>
        <div class="tab" onclick="switchHeatmap('HEATWAVE')">🌡️ Heat</div>
      </div>
      <div id="heatmap-container"></div>
      <div style="display:flex;gap:8px;margin-top:12px;align-items:center;font-size:11px;color:var(--txt2);font-family:var(--mono);">
        <span>Low</span>
        <div style="flex:1;height:8px;border-radius:4px;background:linear-gradient(90deg,#2d6a4f,#95d5b2,#f4a261,#e76f51,#c1121f)"></div>
        <span>Extreme</span>
      </div>
    </div>

    <!-- Disaster Donut Charts -->
    <div class="card">
      <div class="card-title">Risk Level Distribution</div>
      <div class="row row-2" style="gap:16px;margin-bottom:0">
        <div>
          <div style="text-align:center;font-size:12px;color:var(--txt2);margin-bottom:8px">🔥 Fire</div>
          <canvas id="fireDonut" height="160"></canvas>
        </div>
        <div>
          <div style="text-align:center;font-size:12px;color:var(--txt2);margin-bottom:8px">🌊 Flood</div>
          <canvas id="floodDonut" height="160"></canvas>
        </div>
        <div>
          <div style="text-align:center;font-size:12px;color:var(--txt2);margin-bottom:8px">⛰️ Landslide</div>
          <canvas id="landslideDonut" height="160"></canvas>
        </div>
        <div>
          <div style="text-align:center;font-size:12px;color:var(--txt2);margin-bottom:8px">🌡️ Heat Wave</div>
          <canvas id="heatwaveDonut" height="160"></canvas>
        </div>
      </div>
    </div>
  </div>

  <!-- ACTIVE ALERTS -->
  <div class="section-title">Active Red Alerts</div>
  <div class="card card-red" style="margin-bottom:32px">
    <div class="card-title">Emergency Level Alerts — Sorted by Date</div>
    <div class="alerts-list">
      {alert_rows_html}
    </div>
  </div>

  <!-- FEATURE IMPORTANCE -->
  <div class="section-title">ML Model Insights</div>
  <div class="row row-2" style="margin-bottom:32px">
    <div class="card">
      <div class="card-title">Feature Importance — What Drives Each Disaster</div>
      <canvas id="featureChart" height="280"></canvas>
    </div>
    <div class="card">
      <div class="card-title">Model Accuracy Summary</div>
      <div style="display:flex;flex-direction:column;gap:20px;padding-top:8px">
        {"".join([f'''
        <div>
          <div style="display:flex;justify-content:space-between;margin-bottom:6px">
            <span style="font-size:13px;font-weight:600">{em} {name}</span>
            <span style="font-family:var(--mono);font-size:13px;color:var(--acc)">{acc}%</span>
          </div>
          <div style="height:6px;background:var(--bdr);border-radius:3px">
            <div style="height:6px;width:{acc}%;background:linear-gradient(90deg,var(--acc),var(--acc2));border-radius:3px"></div>
          </div>
          <div style="font-size:11px;color:var(--txt2);margin-top:4px;font-family:var(--mono)">{driver}</div>
        </div>''' for em,name,acc,driver in [
          ("🔥","Forest Fire","99.4","Top driver: Humidity (24.6%)"),
          ("🌊","Flood","99.1","Top driver: 3-Day Rain Total (18.7%)"),
          ("⛰️","Landslide","98.8","Top driver: 7-Day Humidity (18.4%)"),
          ("🌡️","Heat Wave","99.3","Top driver: Max Temperature (22.5%)"),
        ]])}
      </div>
    </div>
  </div>

</div>

<!-- FOOTER -->
<div class="footer">
  <span>🏔️ Uttarakhand Disaster Warning System</span>
  <span>NASA POWER MERRA-2 · Random Forest ML · Mar 2025 – Mar 2026</span>
  <span id="gen-time"></span>
</div>

<script>
// ── DATA FROM PYTHON ──
const monthlyData  = {monthly_json};
const gridData     = {grid_json};
const districtData = {district_json};

// ── CLOCK ──
function updateClock(){{
  document.getElementById('clock').textContent = new Date().toLocaleTimeString('en-IN');
}}
setInterval(updateClock, 1000); updateClock();
document.getElementById('gen-time').textContent =
  'Generated: ' + new Date().toLocaleDateString('en-IN');

// ── CHART DEFAULTS ──
Chart.defaults.color = '#8b949e';
Chart.defaults.font.family = 'JetBrains Mono';

// ── MONTHLY RISK CHART ──
const riskCtx = document.getElementById('monthlyRiskChart').getContext('2d');
const monthlyRiskChart = new Chart(riskCtx, {{
  type: 'line',
  data: {{
    labels: monthlyData.map(d => d.LABEL),
    datasets: [
      {{ label:'🔥 Fire',      data: monthlyData.map(d=>d.FIRE),      borderColor:'#ef233c', backgroundColor:'#ef233c11', tension:0.4, fill:true, borderWidth:2, pointRadius:3 }},
      {{ label:'🌊 Flood',     data: monthlyData.map(d=>d.FLOOD),     borderColor:'#3d9bff', backgroundColor:'#3d9bff11', tension:0.4, fill:true, borderWidth:2, pointRadius:3 }},
      {{ label:'⛰️ Landslide', data: monthlyData.map(d=>d.LANDSLIDE), borderColor:'#a855f7', backgroundColor:'#a855f711', tension:0.4, fill:true, borderWidth:2, pointRadius:3 }},
      {{ label:'🌡️ Heat Wave', data: monthlyData.map(d=>d.HEATWAVE),  borderColor:'#f77f00', backgroundColor:'#f77f0011', tension:0.4, fill:true, borderWidth:2, pointRadius:3 }},
    ]
  }},
  options:{{
    responsive:true, maintainAspectRatio:false,
    plugins:{{ legend:{{ position:'top', labels:{{ boxWidth:12, font:{{ size:11 }} }} }} }},
    scales:{{
      y:{{ min:0, max:4, ticks:{{ callback: v=>['Min','Low','Mod','High','Ext'][v]||v, stepSize:1 }}, grid:{{ color:'#21262d' }} }},
      x:{{ grid:{{ color:'#21262d' }}, ticks:{{ maxRotation:30 }} }}
    }}
  }}
}});

// ── WEATHER CHART ──
const wCtx = document.getElementById('weatherChart').getContext('2d');
new Chart(wCtx, {{
  type:'bar',
  data:{{
    labels: monthlyData.map(d=>d.LABEL),
    datasets:[
      {{ label:'Rainfall (mm/day)', data:monthlyData.map(d=>d.RAIN), backgroundColor:'#3d9bff55', borderColor:'#3d9bff', borderWidth:1, yAxisID:'y' }},
      {{ label:'Temperature (°C)',  data:monthlyData.map(d=>d.TEMP), type:'line', borderColor:'#ef233c', backgroundColor:'transparent', borderWidth:2, tension:0.4, yAxisID:'y2', pointRadius:3 }},
    ]
  }},
  options:{{
    responsive:true, maintainAspectRatio:false,
    plugins:{{ legend:{{ position:'top', labels:{{ boxWidth:12, font:{{ size:11 }} }} }} }},
    scales:{{
      y:{{ position:'left', title:{{ display:true, text:'Rain mm/day' }}, grid:{{ color:'#21262d' }} }},
      y2:{{ position:'right', title:{{ display:true, text:'Temp °C' }}, grid:{{ drawOnChartArea:false }} }},
      x:{{ grid:{{ color:'#21262d' }}, ticks:{{ maxRotation:30 }} }}
    }}
  }}
}});

// ── TAB SWITCH ──
function switchTab(tab){{
  document.querySelectorAll('.tabs .tab').forEach((t,i)=>{{
    if(t.closest('.card') === document.getElementById('chart-'+tab)?.closest('.card') ||
       t.closest('.card') === document.getElementById('chart-risk')?.closest('.card')){{
      t.classList.toggle('active', (tab==='risk'&&i===0)||(tab==='weather'&&i===1));
    }}
  }});
  document.getElementById('chart-risk').style.display    = tab==='risk'    ? 'block' : 'none';
  document.getElementById('chart-weather').style.display = tab==='weather' ? 'block' : 'none';
}}

// ── HEATMAP ──
const riskColorFn = s => {{
  if(s>=3.5) return '#c1121f';
  if(s>=2.5) return '#e76f51';
  if(s>=1.5) return '#f4a261';
  if(s>=0.5) return '#95d5b2';
  return '#2d6a4f';
}};

function buildHeatmap(field){{
  const container = document.getElementById('heatmap-container');
  const lats = [...new Set(gridData.map(d=>d.LAT))].sort((a,b)=>b-a);
  const lons = [...new Set(gridData.map(d=>d.LON))].sort((a,b)=>a-b);
  const cellSize = Math.floor((container.offsetWidth||400) / lons.length);
  container.style.cssText = `display:grid;grid-template-columns:repeat(${{lons.length}},1fr);gap:3px;`;
  container.innerHTML = '';
  lats.forEach(lat=>{{
    lons.forEach(lon=>{{
      const cell = gridData.find(d=>d.LAT===lat&&d.LON===lon);
      const score = cell ? (cell[field]||0) : 0;
      const div = document.createElement('div');
      div.className = 'hm-cell';
      div.style.background = riskColorFn(score);
      div.style.minHeight = '36px';
      div.title = `LAT:${{lat}} LON:${{lon}} | ${{field}}: ${{score.toFixed(2)}}`;
      div.textContent = score.toFixed(1);
      container.appendChild(div);
    }});
  }});
}}
buildHeatmap('MAX');

function switchHeatmap(field){{
  document.querySelectorAll('#heatmap-container').forEach(()=>{{}});
  event.target.closest('.tabs').querySelectorAll('.tab').forEach(t=>t.classList.remove('active'));
  event.target.classList.add('active');
  buildHeatmap(field);
}}

// ── DONUT CHARTS ──
function makeDonut(id, data, colors){{
  new Chart(document.getElementById(id).getContext('2d'),{{
    type:'doughnut',
    data:{{ labels:['Minimal','Low','Moderate','High','Extreme'], datasets:[{{ data, backgroundColor:colors, borderWidth:0 }}] }},
    options:{{ responsive:true, plugins:{{ legend:{{ display:false }} }}, cutout:'65%' }}
  }});
}}

const donutColors = ['#2d6a4f','#95d5b2','#f4a261','#e76f51','#c1121f'];

// Calculate distributions from district data
function getRiskDist(field){{
  const vals = districtData.map(d=>d[field]);
  return [
    vals.filter(v=>v<0.5).length,
    vals.filter(v=>v>=0.5&&v<1.5).length,
    vals.filter(v=>v>=1.5&&v<2.5).length,
    vals.filter(v=>v>=2.5&&v<3.5).length,
    vals.filter(v=>v>=3.5).length,
  ];
}}

makeDonut('fireDonut',      getRiskDist('FIRE'),      donutColors);
makeDonut('floodDonut',     getRiskDist('FLOOD'),     donutColors);
makeDonut('landslideDonut', getRiskDist('LANDSLIDE'), donutColors);
makeDonut('heatwaveDonut',  getRiskDist('HEATWAVE'),  donutColors);

// ── FEATURE IMPORTANCE CHART ──
const fCtx = document.getElementById('featureChart').getContext('2d');
new Chart(fCtx,{{
  type:'bar',
  data:{{
    labels:['Humidity','Max Temp','7D Humidity','Temperature','3D Rain','7D Rain','Rainfall','Soil Wet','Max Temp','Temp'],
    datasets:[
      {{ label:'🔥 Fire',      data:[24.6,9.8,9.2,7.3,0,0,0,0,0,0],      backgroundColor:'#ef233c88', borderColor:'#ef233c', borderWidth:1 }},
      {{ label:'🌊 Flood',     data:[0,0,6.2,0,18.7,16.7,13.4,9.6,0,0],  backgroundColor:'#3d9bff88', borderColor:'#3d9bff', borderWidth:1 }},
      {{ label:'⛰️ Landslide', data:[7.1,0,18.4,0,0,10.6,0,15.1,0,0],    backgroundColor:'#a855f788', borderColor:'#a855f7', borderWidth:1 }},
      {{ label:'🌡️ Heat Wave', data:[0,22.5,4.7,14.7,0,0,0,0,22.5,14.7], backgroundColor:'#f77f0088', borderColor:'#f77f00', borderWidth:1 }},
    ]
  }},
  options:{{
    responsive:true, maintainAspectRatio:false, indexAxis:'y',
    plugins:{{ legend:{{ position:'top', labels:{{ boxWidth:12, font:{{ size:11 }} }} }} }},
    scales:{{
      x:{{ title:{{ display:true, text:'Importance %' }}, grid:{{ color:'#21262d' }} }},
      y:{{ grid:{{ color:'#21262d' }} }}
    }}
  }}
}});

// ── DISTRICT SELECT ──
function selectDistrict(name){{
  document.querySelectorAll('.dcard').forEach(c=>c.classList.remove('active'));
  const card = document.getElementById('dc-'+name.replace(' ','-'));
  if(card) card.classList.add('active');
}}
</script>
</body>
</html>"""

# ── Save ──
os.makedirs("dashboard", exist_ok=True)
path = "dashboard/uttarakhand_dashboard.html"
with open(path, "w", encoding="utf-8") as f:
    f.write(html)

size_mb = os.path.getsize(path) / (1024*1024)
print(f"\n{'='*60}")
print(f"  STEP 6 COMPLETE ✅")
print(f"  Dashboard : {path}  ({size_mb:.1f} MB)")
print(f"{'='*60}")
print("""
Download & open in browser:
  from google.colab import files
  files.download('dashboard/uttarakhand_dashboard.html')
""")

HTML components built ✅
District cards : 20
Alert rows     : 8

  STEP 6 COMPLETE ✅
  Dashboard : dashboard/uttarakhand_dashboard.html  (0.0 MB)

Download & open in browser:
  from google.colab import files
  files.download('dashboard/uttarakhand_dashboard.html')



In [ ]:
from google.colab import files
files.download('dashboard/uttarakhand_dashboard.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>